# CS781 Phase 1 — four from-scratch legal architectures

Every source file is directly readable below. Run this in a local or remote
**Jupyter notebook with two visible NVIDIA GPUs, about 24 GB each**. Accept access
to the official dataset. Authenticate with an existing Hugging Face login, the
`HF_TOKEN` environment variable, or the masked prompt in the authentication cell.
Default: profile all four models, set measured budgets, then train on both GPUs.
Use `MODE='profile'` to stop after calibration; it creates no submission ZIPs.

The provided official BERTScore evaluator selects checkpoints and decoders.
Its pretrained scoring model is used for evaluation only. Phase 1 models start
from random weights and use only the released legal corpus.


## 1. Write the visible source files
Run these standard Jupyter `%%writefile` cells before the workflow.


In [ ]:
import os, sys
from pathlib import Path
start_directory = Path.cwd().resolve()
if start_directory.name == 'cs781_phase1' and (start_directory/'phase1').is_dir():
    start_directory = start_directory.parent
WORK = Path(os.environ.get('CS781_WORK_DIR', str(start_directory))).expanduser().resolve()
PROJECT = WORK/'cs781_phase1'
PROJECT.mkdir(parents=True, exist_ok=True)
for directory in ['phase1','configs','evaluators']:
    (PROJECT/directory).mkdir(exist_ok=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)
print('Use a fresh kernel after editing source cells or changing CUDA visibility.')


### `phase1/__init__.py`


In [ ]:
%%writefile phase1/__init__.py
"""From-scratch CS781 Phase 1 experiments; no external model weights."""


### `phase1/common.py`


In [ ]:
%%writefile phase1/common.py
"""Small provenance, serialization, and reproducibility utilities."""
import hashlib
import json
import os
import random
import uuid
import warnings
from pathlib import Path


def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value
    return result


def read_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle, object_pairs_hook=unique_object)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    os.replace(temporary, path)


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seed_everything(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def document_hash(text):
    # Whitespace-normalized exact duplicates share the split across sources.
    return hashlib.sha256(" ".join(text.split()).encode()).hexdigest()


def is_validation(text, seed=42, fraction=0.02):
    key = hashlib.sha256(f"{seed}:{document_hash(text)}".encode()).digest()
    return int.from_bytes(key[:8], "big") / 2**64 < fraction


def quarantine_directory(path, reason):
    """Preserve incomplete work under a new name before restarting."""
    path = Path(path)
    if not path.exists():
        return None
    saved = path.with_name(path.name + '.incomplete-' + uuid.uuid4().hex[:10])
    path.rename(saved)
    warnings.warn(f"{reason}; preserved incomplete files at {saved}", RuntimeWarning)
    return saved


def directory_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file()) if path.exists() else 0


### `phase1/data.py`


In [ ]:
%%writefile phase1/data.py
"""Stream official legal JSON and create reproducible disk-backed token corpora."""
import argparse
import itertools
import json
import math
import os
import shutil
import sqlite3
from pathlib import Path
import warnings
import ijson
import numpy as np
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from .common import document_hash, file_hash, is_validation, read_json, write_json, quarantine_directory

SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>"]
REPO = "Exploration-Lab/CS-781-Capstone"
PREFIX = "Phase_1_Pretraining/"


def records(path):
    """Accept the starter's top-level list, an ID-keyed object, or JSONL."""
    path = Path(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        with path.open(encoding="utf-8") as handle:
            for index, line in enumerate(handle):
                if line.strip():
                    record = json.loads(line)
                    if not isinstance(record, dict) or not isinstance(record.get("text"), str):
                        raise ValueError(f"Invalid JSONL record {index}: expected string text")
                    yield record
        return
    with path.open("rb") as handle:
        first = b""
        while not first.strip():
            first = handle.read(1)
            if not first:
                raise ValueError(f"Empty JSON file: {path}")
        handle.seek(0)
        if first == b"[":
            iterator = ijson.items(handle, "item")
        elif first == b"{":
            iterator = (value for _, value in ijson.kvitems(handle, ""))
        else:
            raise ValueError(f"Unsupported JSON container in {path}")
        for index, record in enumerate(iterator):
            if not isinstance(record, dict) or not isinstance(record.get("text"), str):
                raise ValueError(f"{path}, record {index}: expected {{'text': string}}; inspect real schema before training")
            if record["text"].strip():
                yield record


def resolve_official(output_dir, sources, revision="main", test_path=None, raw_dir=None):
    """Resolve commit and real filenames first; never guess a test filename."""
    from huggingface_hub import HfApi, hf_hub_download
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    info = api.dataset_info(REPO, revision=revision, files_metadata=True)
    sizes = {item.rfilename: item.size or 0 for item in info.siblings}
    train_files = [PREFIX + "Train/" + item for item in sources]
    missing = set(train_files) - set(sizes)
    if missing:
        raise ValueError(f"Training files absent at {info.sha}: {sorted(missing)}")
    candidates = sorted(name for name in sizes if name.startswith(PREFIX + "Test/") and name.lower().endswith(".json"))
    if test_path is None:
        if len(candidates) != 1:
            raise ValueError(f"Choose --test-file from actual repository paths: {candidates}")
        test_path = candidates[0]
    elif test_path not in candidates:
        raise ValueError(f"Test path absent; available: {candidates}")
    selected = train_files + [test_path]
    download_dir = Path(raw_dir) if raw_dir is not None else output / "raw"
    download_dir.mkdir(parents=True, exist_ok=True)
    needed = sum(sizes[name] for name in selected if not (download_dir / name).exists())
    reserve = 4 * 1024**3  # token files, checkpoints, optimizer, and scratch
    if needed + reserve > shutil.disk_usage(download_dir).free:
        raise OSError(f"Selected raw files need {needed / 1024**3:.2f} GiB plus a 4 GiB reserve; reduce sources or attach local data")
    paths = [hf_hub_download(REPO, name, repo_type="dataset", revision=info.sha,
                            token=os.environ.get("HF_TOKEN"), local_dir=download_dir)
             for name in selected]
    metadata = {"repo": REPO, "revision": info.sha,
                "files": [{"path": name, "bytes": sizes[name]} for name in selected],
                "train_paths": paths[:-1], "test_path": paths[-1]}
    write_json(output / "source_manifest.json", metadata)
    return metadata


def inspect(paths, limit=3):
    for path in paths:
        print(f"FILE {path}")
        for record in itertools.islice(records(path), limit):
            print({"keys": list(record), "words": len(record["text"].split()),
                   "text_preview": record["text"][:240]})


def train_tokenizer(paths, destination, seed=42, val_fraction=0.02,
                    vocab_size=24000, max_chars=20_000_000):
    if vocab_size < 260:
        raise ValueError("Byte BPE needs at least 256 bytes plus four special tokens")
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                   special_tokens=SPECIAL,
                                   initial_alphabet=pre_tokenizers.ByteLevel.alphabet())

    def text_iterator():
        total = 0
        iterators = [iter(records(path)) for path in paths]
        # Round-robin avoids training the tokenizer entirely on the first source.
        while iterators and total < max_chars:
            active = []
            for iterator in iterators:
                try:
                    text = next(iterator)["text"]
                except StopIteration:
                    continue
                active.append(iterator)
                if not is_validation(text, seed, val_fraction):
                    part = text[:max_chars-total]
                    total += len(part)
                    yield part
                if total >= max_chars:
                    break
            iterators = active
        if not total:
            raise ValueError("No training-only tokenizer text available")

    tokenizer.train_from_iterator(text_iterator(), trainer=trainer)
    tokenizer.save(str(destination))
    return tokenizer


def quotas(total, weights):
    if any(weight <= 0 for weight in weights):
        raise ValueError("All source weights must be positive")
    norm = sum(weights)
    result = [int(total * weight / norm) for weight in weights]
    result[-1] += total - sum(result)
    return result


def record_batches(path, size=32, max_chars=2_000_000):
    batch, chars = [], 0
    for record in records(path):
        batch.append(record)
        chars += len(record['text'])
        if len(batch) >= size or chars >= max_chars:
            yield batch
            batch, chars = [], 0
    if batch:
        yield batch


def _prepare(paths, output_dir, train_tokens=50_000_000, val_tokens=1_000_000,
            weights=None, seed=42, val_fraction=0.02, vocab_size=24000,
            tokenizer_chars=20_000_000, generation_examples=64, source_manifest=None,
            tokenizer_path=None, tokenize_batch_size=32):
    output = Path(output_dir)
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"Refusing to overwrite preprocessing: {output}; reuse it or choose a new directory")
    output.mkdir(parents=True, exist_ok=True)
    if not paths or not 0 < val_fraction < 1 or min(train_tokens, val_tokens) <= 0:
        raise ValueError("Need sources, positive token budgets, and val_fraction in (0,1)")
    if tokenize_batch_size < 1 or generation_examples < 0 or tokenizer_chars < 1:
        raise ValueError('Invalid tokenizer batch/sample settings')
    paths = [str(Path(path).resolve()) for path in paths]
    weights = weights or [1] * len(paths)
    if len(weights) != len(paths):
        raise ValueError("Source/weight count mismatch")
    if tokenizer_path is None:
        tokenizer = train_tokenizer(paths, output / "tokenizer.json", seed, val_fraction,
                                    vocab_size, tokenizer_chars)
    else:
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
        if any(tokenizer.token_to_id(token) is None for token in SPECIAL):
            raise ValueError('Reused tokenizer must define pad, bos, eos and unk special tokens')
        shutil.copyfile(tokenizer_path, output / 'tokenizer.json')
    dtype = np.dtype("uint16" if tokenizer.get_vocab_size() <= 65536 else "uint32")
    eos = tokenizer.token_to_id("<eos>")
    train_quota, val_quota = quotas(train_tokens, weights), quotas(val_tokens, weights)
    db = sqlite3.connect(output / "documents.sqlite")
    db.execute("CREATE TABLE documents (hash TEXT PRIMARY KEY, split TEXT, source INTEGER, tokens INTEGER)")
    counts = {"train": 0, "validation": 0, "duplicates": 0, "documents": 0}
    per_source, examples = [], {}
    example_quotas = quotas(generation_examples, [1] * len(paths))
    reservoirs = [[] for _ in paths]
    eligible = [0 for _ in paths]
    try:
        with (output / "train.bin").open("wb") as train_handle, (output / "validation.bin").open("wb") as val_handle:
            for index, path in enumerate(paths):
                used = {"train": 0, "validation": 0}
                target = {"train": train_quota[index], "validation": val_quota[index]}
                reservoir_rng = np.random.default_rng(seed + 10000 + index)
                source_documents = {'train': 0, 'validation': 0}
                for batch in record_batches(path, tokenize_batch_size):
                    selected = [record for record in batch if used['validation' if is_validation(record['text'], seed, val_fraction) else 'train'] < target['validation' if is_validation(record['text'], seed, val_fraction) else 'train']]
                    encodings = tokenizer.encode_batch([record['text'] for record in selected])
                    for record, encoding in zip(selected, encodings):
                        text = record["text"]
                        key = document_hash(text)
                        split = "validation" if is_validation(text, seed, val_fraction) else "train"
                        if used[split] >= target[split]:
                            continue
                        try:
                            db.execute("INSERT INTO documents VALUES (?,?,?,?)", (key, split, index, 0))
                        except sqlite3.IntegrityError:
                            counts["duplicates"] += 1
                            continue
                        words = text.split() if split == "validation" else None
                        if words is not None and len(words) >= 640:
                            eligible[index] += 1
                            start = int(key[:8], 16) % (len(words)-640+1)
                            example = (key, {"text": " ".join(words[start:start+512]),
                                             "reference": " ".join(words[start+512:start+640]),
                                             "source": Path(path).name, 'source_id': index})
                            if len(reservoirs[index]) < example_quotas[index]:
                                reservoirs[index].append(example)
                            elif example_quotas[index]:
                                slot = int(reservoir_rng.integers(0, eligible[index]))
                                if slot < example_quotas[index]:
                                    reservoirs[index][slot] = example
                        tokens = (encoding.ids + [eos])[:target[split]-used[split]]
                        np.asarray(tokens, dtype=dtype).tofile(val_handle if split == "validation" else train_handle)
                        used[split] += len(tokens)
                        counts[split] += len(tokens)
                        counts["documents"] += 1
                        source_documents[split] += 1
                        db.execute("UPDATE documents SET tokens=? WHERE hash=?", (len(tokens), key))
                        if counts["documents"] % 1000 == 0:
                            db.commit()
                            print({"source": index, "tokens": dict(counts)}, flush=True)
                    if all(used[part] >= target[part] for part in used):
                        break
                per_source.append({"path": path, "bytes": Path(path).stat().st_size,
                                   "requested": target, "observed": used,
                                   'documents': source_documents,
                                   'eligible_generation_documents': eligible[index],
                                   'generation_examples': len(reservoirs[index])})
        db.commit()
        if not counts["train"] or not counts["validation"]:
            raise ValueError("Empty training/validation split; add data or change split fraction")
    finally:
        db.close()
    examples = dict(pair for reservoir in reservoirs for pair in reservoir)
    metadata = {"seed": seed, "val_fraction": val_fraction, "split_unit": "whitespace-normalized document hash",
                "vocab_size": tokenizer.get_vocab_size(), "dtype": dtype.name,
                "counts": counts, "sources": per_source,
                "official_source_manifest": source_manifest,
                "tokenizer_sha256": file_hash(output / "tokenizer.json"),
                "train_sha256": file_hash(output / "train.bin"),
                "validation_sha256": file_hash(output / "validation.bin"),
                'generation_sampling': 'balanced per-source reservoir over eligible held-out documents',
                'tokenize_batch_size': tokenize_batch_size,
                "generation_examples": len(examples), "complete": True}
    write_json(output / "validation_examples.json", examples)
    write_json(output / "metadata.json", metadata)
    for source in per_source:
        if source["requested"] != source["observed"]:
            print("WARNING: exhausted source before requested budgets:", source)
    if not examples:
        print("WARNING: no held-out 640-word document; generation evaluation unavailable")
    return metadata


def prepare(paths, output_dir, train_tokens=50_000_000, val_tokens=1_000_000,
            weights=None, seed=42, val_fraction=0.02, vocab_size=24000,
            tokenizer_chars=20_000_000, generation_examples=64, source_manifest=None,
            tokenizer_path=None, tokenize_batch_size=32, reuse=False,
            recover_incomplete=True):
    """Publish only completed preprocessing; preserve interrupted stages."""
    output = Path(output_dir)
    output.parent.mkdir(parents=True, exist_ok=True)
    arguments = dict(train_tokens=train_tokens, val_tokens=val_tokens, weights=weights,
                     seed=seed, val_fraction=val_fraction, vocab_size=vocab_size,
                     tokenizer_chars=tokenizer_chars, generation_examples=generation_examples,
                     source_manifest=source_manifest,
                     tokenizer_path=str(Path(tokenizer_path).resolve()) if tokenizer_path else None,
                     tokenize_batch_size=tokenize_batch_size)
    request = {'paths': [str(Path(p).resolve()) for p in paths],
               'source_files': [{'path':str(Path(p).resolve()), 'bytes':Path(p).stat().st_size,
                                 'mtime_ns':Path(p).stat().st_mtime_ns} for p in paths],
               'reused_tokenizer_sha256':file_hash(tokenizer_path) if tokenizer_path else None, **arguments}
    metadata_path = output / 'metadata.json'
    if metadata_path.exists() and read_json(metadata_path).get('complete'):
        if reuse and read_json(output / 'preparation_request.json') == request:
            return read_json(metadata_path)
        raise FileExistsError(f'Completed preprocessing exists: {output}; reuse matching settings or choose a new directory')
    if output.exists() and any(output.iterdir()):
        if not recover_incomplete:
            raise FileExistsError(f'Incomplete preprocessing at {output}')
        quarantine_directory(output, 'Restarting incomplete preprocessing')
    stage = output.with_name(output.name + '.partial')
    if stage.exists():
        if not recover_incomplete:
            raise FileExistsError(f'Incomplete preprocessing stage at {stage}')
        quarantine_directory(stage, 'Restarting interrupted preprocessing stage')
    try:
        result = _prepare(paths, stage, **arguments)
        write_json(stage / 'preparation_request.json', request)
        # The final directory never exposes half-written token files.
        os.replace(stage, output)
        return result
    except Exception as error:
        if stage.exists():
            write_json(stage / 'failure.json', {'error': str(error), 'status': 'incomplete'})
        raise


class TokenCorpus:
    def __init__(self, directory, split="train"):
        directory = Path(directory)
        self.metadata = read_json(directory / "metadata.json")
        if not self.metadata.get("complete"):
            raise ValueError("Incomplete preprocessing")
        self.tokens = np.memmap(directory / f"{split}.bin", dtype=self.metadata["dtype"], mode="r")

    def sample(self, rng, batch_size, length):
        if len(self.tokens) < length:
            raise ValueError(f"Token split ({len(self.tokens)}) is shorter than a block ({length})")
        starts = rng.integers(0, len(self.tokens)-length+1, size=batch_size)
        return np.stack([self.tokens[start:start+length].astype(np.int64) for start in starts])


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    fetch = sub.add_parser("fetch")
    fetch.add_argument("--output", required=True)
    fetch.add_argument("--sources", nargs="+", default=["train_supreme_court.json"])
    fetch.add_argument("--revision", default="main")
    fetch.add_argument("--test-file")
    fetch.add_argument('--raw-dir', help='Temporary raw download location, e.g. /tmp/cs781_raw')
    show = sub.add_parser("inspect")
    show.add_argument("paths", nargs="+")
    prep = sub.add_parser("prepare")
    prep.add_argument("--paths", nargs="+", required=True)
    prep.add_argument("--output", required=True)
    prep.add_argument("--train-tokens", type=int, default=50_000_000)
    prep.add_argument("--val-tokens", type=int, default=1_000_000)
    prep.add_argument("--weights", type=float, nargs="+")
    prep.add_argument("--seed", type=int, default=42)
    prep.add_argument("--val-fraction", type=float, default=0.02)
    prep.add_argument("--vocab-size", type=int, default=24000)
    prep.add_argument("--tokenizer-chars", type=int, default=20_000_000)
    prep.add_argument("--generation-examples", type=int, default=64)
    prep.add_argument("--source-manifest")
    prep.add_argument('--tokenizer-path', help='Reuse your course/pilot tokenizer with the same document split')
    prep.add_argument('--tokenize-batch-size', type=int, default=32)
    prep.add_argument('--reuse', action='store_true')
    args = vars(parser.parse_args())
    command = args.pop("command")
    if command == "fetch":
        args["output_dir"] = args.pop("output")
        args["test_path"] = args.pop("test_file")
        print(resolve_official(**args))
    elif command == "inspect":
        inspect(args["paths"])
    else:
        args["output_dir"] = args.pop("output")
        if args["source_manifest"]:
            args["source_manifest"] = read_json(args["source_manifest"])
        print(prepare(**args))


if __name__ == "__main__":
    main()


### `phase1/evaluate.py`


In [ ]:
%%writefile phase1/evaluate.py
"""Comparable held-out continuation loss plus lexical diagnostics.

No pretrained BERT evaluator is downloaded here. Use the supplied official
scorer for BERTScore; lexical scores do not reproduce the primary metric.
"""
import argparse
from pathlib import Path
import math
import torch
from torch.nn import functional as F
from .common import read_json, write_json
from .generate import load_checkpoint
from .submission import validate
from .train import amp_context, precision


def rouge_l(prediction, reference):
    a, b = prediction.split(), reference.split()
    row = [0] * (len(b)+1)
    for word in a:
        nxt = [0]
        for j, other in enumerate(b):
            nxt.append(row[j]+1 if word == other else max(row[j+1], nxt[-1]))
        row = nxt
    return 2 * row[-1] / (len(a)+len(b)) if a or b else 1.0


@torch.inference_mode()
def conditional_nll(model, tokenizer, examples, dtype=None):
    """Score exactly the same reference tokens conditioned on each 512-word prompt.

    Prompt and continuation are encoded separately, consistently for all four
    architectures. This boundary convention is logged and not a Codabench score.
    """
    device = next(model.parameters()).device
    model.eval()
    summed, count = 0.0, 0
    for record in examples.values():
        prompt = tokenizer.encode(record["text"].rstrip()).ids
        target = tokenizer.encode(" " + record["reference"].lstrip()).ids
        if not prompt or not target:
            raise ValueError("Empty prompt/reference token sequence")
        source = torch.tensor([prompt], device=device)
        labels = torch.tensor([target], device=device)
        with amp_context(device, dtype):
            if model.config.architecture == "encoder_decoder":
                inputs = torch.tensor([[tokenizer.token_to_id("<bos>")] + target[:-1]], device=device)
                logits, _ = model(inputs, source_ids=source)
            else:
                inputs = torch.tensor([prompt + target[:-1]], device=device)
                logits, _ = model(inputs)
                logits = logits[:, len(prompt)-1:]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), reduction="sum")
        summed += loss.item()
        count += len(target)
    mean = summed/count
    return {"conditional_nll": mean, "conditional_perplexity": math.exp(min(mean, 700)),
            "reference_tokens": count, "examples": len(examples),
            "boundary": "prompt.rstrip() and space-prefixed reference encoded separately"}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--examples", required=True)
    parser.add_argument("--checkpoint")
    parser.add_argument("--predictions")
    parser.add_argument("--output", required=True)
    args = parser.parse_args()
    if not args.checkpoint and not args.predictions:
        parser.error("Provide checkpoint and/or generated predictions")
    examples = read_json(args.examples)
    if not examples or any(not isinstance(row.get("reference"), str) for row in examples.values()):
        raise ValueError("Need nonempty held-out examples with reference fields")
    metrics = {}
    if args.checkpoint:
        model, tokenizer, _ = load_checkpoint(args.checkpoint)
        _, dtype = precision(next(model.parameters()).device)
        metrics.update(conditional_nll(model, tokenizer, examples, dtype))
    if args.predictions:
        predictions = read_json(args.predictions)
        validate(predictions, examples)
        scores = [rouge_l(predictions[key]["text"], row["reference"]) for key, row in examples.items()]
        metrics["rouge_l_whitespace_f1"] = sum(scores) / len(scores)
        metrics["repeated_trigram_fraction"] = sum(
            1 - len(set(zip(words, words[1:], words[2:]))) / max(1, len(words)-2)
            for words in [row["text"].split() for row in predictions.values()]) / len(predictions)
    metrics["official_bertscore"] = None
    metrics["official_metric_note"] = "Run the supplied course scorer; lexical diagnostics are not official BERTScore"
    write_json(args.output, metrics)
    print(metrics)


if __name__ == "__main__":
    main()


### `phase1/experiments.py`


In [ ]:
%%writefile phase1/experiments.py
"""Dry-run or concurrently execute the four-architecture manifest."""
import argparse
import csv
import json
from pathlib import Path
import shlex
import subprocess
import sys
from .common import read_json
from .train import resolve, check_resume
from .scheduler import run_jobs


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    matrix = sub.add_parser("matrix")
    matrix.add_argument("--data", required=True)
    matrix.add_argument("--output", required=True)
    matrix.add_argument("--configs", nargs="+", default=[f"configs/{name}.json" for name in
                              ["dense_gpt", "gqa_local", "encoder_decoder", "recurrent_hybrid"]])
    matrix.add_argument("--execute", action="store_true", help="Otherwise only print a dry-run")
    matrix.add_argument("--resume", action="store_true")
    matrix.add_argument("--gpus", nargs="*")
    matrix.add_argument("--allow-code-change", action="store_true")
    matrix.add_argument("--max-hours-per-run", type=float)
    ledger = sub.add_parser("ledger")
    ledger.add_argument("--runs", required=True)
    ledger.add_argument("--output", required=True)
    args = parser.parse_args()
    if args.command == "matrix":
        run_ids = set()
        commands = []
        for config in args.configs:
            row = resolve(config, args.data)
            if not 50_000_000 <= row["parameters"] <= 300_000_000:
                raise ValueError(f"Out-of-range competition model: {row['run_id']} ({row['parameters']:,})")
            if row["run_id"] in run_ids:
                raise ValueError(f"Duplicate run_id: {row['run_id']}")
            run_ids.add(row["run_id"])
            output = Path(args.output) / row["run_id"]
            status = read_json(output / "status.json") if (output / "status.json").exists() else {}
            if (output / "resolved_config.json").exists() and ((output / "last.pt").exists() or status.get("status") == "completed"):
                previous = read_json(output / "resolved_config.json")
                check_resume(previous, row, args.allow_code_change)
            if status.get("status") == "completed" and args.resume:
                print(f"SKIP completed run {row['run_id']}")
                continue
            if output.exists() and any(output.iterdir()) and (output / 'last.pt').exists() and not args.resume:
                raise FileExistsError(f"Run exists: {output}; use --resume or a new output root")
            command = [sys.executable, "-m", "phase1.train", "--config", config,
                       "--data", args.data, "--output", str(output)]
            if args.resume and (output / "last.pt").exists():
                command.append("--resume")
            if args.allow_code_change:
                command.append("--allow-code-change")
            if args.max_hours_per_run is not None:
                command += ["--max-hours", str(args.max_hours_per_run)]
            print({"run_id": row["run_id"], "parameters": row["parameters"],
                   "optimizer_steps": row["total_steps"], "command": shlex.join(command)}, flush=True)
            commands.append((command, output))
        # Resolve every manifest row before launching the first expensive run.
        if args.execute:
            jobs = [{'id': output.name, 'argv': command,
                     'log': str(output.parent / (output.name + '.console.log'))}
                    for command, output in commands]
            run_jobs(jobs, args.gpus, status_path=Path(args.output)/'scheduler_status.json')
    else:
        rows = []
        for path in sorted(Path(args.runs).glob("*/resolved_config.json")):
            config = read_json(path)
            root = path.parent
            status = read_json(root / "status.json") if (root / "status.json").exists() else {}
            selection = read_json(root / "selection.json") if (root / "selection.json").exists() else {}
            local = read_json(root / "local_metrics.json") if (root / "local_metrics.json").exists() else {}
            row = {"run_id": config["run_id"], "status": status.get("status", "unknown"),
                   "architecture": config["model"]["architecture"], "parameters": config["parameters"],
                   "seed": config["training"]["seed"], "input_tokens_seen": status.get("input_tokens_seen"),
                   "target_tokens_seen": status.get("target_tokens_seen"),
                   "best_validation_loss": status.get("best_validation_loss"),
                   "conditional_nll": local.get("conditional_nll"),
                   "rouge_l_whitespace_f1": local.get("rouge_l_whitespace_f1"),
                   "official_bertscore": selection.get("official_bertscore"),
                   "selected_checkpoint": selection.get("checkpoint"), "selected_decoding": json.dumps(selection.get("decoding")),
                   "train_seconds": status.get("train_seconds"), "checkpoint": status.get("best_checkpoint"),
                   "data_fingerprint": config["data_fingerprint"], "code_fingerprint": config["code_fingerprint"]}
            rows.append(row)
        if not rows:
            raise ValueError("No runs found")
        if len({row["run_id"] for row in rows}) != len(rows):
            raise ValueError("Duplicate run IDs in ledger")
        Path(args.output).parent.mkdir(parents=True, exist_ok=True)
        with open(args.output, "w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print(f"Saved {len(rows)} ledger rows to {args.output}")


if __name__ == "__main__":
    main()


### `phase1/generate.py`


In [ ]:
%%writefile phase1/generate.py
"""Cached length-bucketed generation, deterministic short-output retries and resume."""
import argparse
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
import time
from pathlib import Path
import torch
from tokenizers import Tokenizer
from .common import file_hash, fingerprint, read_json, write_json
from .models import build_model
from .submission import load_test, package
from .train import amp_context, precision


class ShortGenerationError(ValueError):
    """The decoder exhausted its budget or stalled before a complete continuation."""


@dataclass
class DecodingConfig:
    words: int = 128
    max_new_tokens: int = 768
    temperature: float = 0.3
    top_p: float = 0.95
    top_k: int = 0
    repetition_penalty: float = 1.1
    no_repeat_ngram_size: int = 0  # Token n-grams; tune against held-out BERTScore.
    stall_tokens: int = 64
    seed: int = 42

    def validate(self):
        if self.words <= 0 or self.max_new_tokens <= 0 or self.temperature < 0:
            raise ValueError('Invalid generation word/token/temperature setting')
        if not 0 < self.top_p <= 1 or self.top_k < 0 or self.repetition_penalty < 1:
            raise ValueError('Invalid top-p/top-k/repetition setting')
        if self.no_repeat_ngram_size < 0 or self.stall_tokens < 1:
            raise ValueError('Invalid n-gram/stall setting')


def select_token(logits, history, forbidden_ids, config, generator=None):
    logits = logits.float().clone()
    if history and config.repetition_penalty != 1:
        ids = torch.tensor(sorted(set(history)), device=logits.device)
        logits[ids] = torch.where(logits[ids] < 0, logits[ids]*config.repetition_penalty,
                                  logits[ids]/config.repetition_penalty)
    logits[forbidden_ids] = -torch.inf
    n = config.no_repeat_ngram_size
    if n and len(history) >= n-1:
        prefix = tuple(history[-(n-1):]) if n > 1 else ()
        blocked = {history[i+n-1] for i in range(len(history)-n+1)
                   if tuple(history[i:i+n-1]) == prefix}
        if blocked:
            logits[list(blocked)] = -torch.inf
    if not torch.isfinite(logits).any():
        raise FloatingPointError('No finite next-token logit after generation filters')
    if config.temperature == 0:
        return int(logits.argmax())
    logits /= config.temperature
    if config.top_k:
        threshold = logits.topk(min(config.top_k, logits.numel())).values[-1]
        logits[logits < threshold] = -torch.inf
    values, indices = logits.sort(descending=True)
    probabilities = values.softmax(-1)
    values[probabilities.cumsum(-1)-probabilities > config.top_p] = -torch.inf
    sample = torch.multinomial(values.softmax(-1), 1, generator=generator)
    return int(indices[sample])


@torch.inference_mode()
def continuation_batch(model, tokenizer, prompts, config, device, dtype=None, seeds=None):
    """Identical token lengths prevent padding entering attention or LSTM state.

    Return successes and ShortGenerationError objects per row. Other errors
    propagate. Finished rows stay in the cache until their peers finish.
    """
    config.validate()
    model.eval()
    encoded = tokenizer.encode_batch([p.rstrip() for p in prompts])
    lengths = [len(item.ids) for item in encoded]
    if not lengths or not min(lengths) or len(set(lengths)) != 1:
        raise ValueError('A batch requires nonempty prompts of identical token length')
    length = lengths[0]
    if length > model.config.max_context:
        raise ValueError(f'Prompt uses {length} tokens; context is {model.config.max_context}; no silent truncation')
    seq2seq = model.config.architecture == 'encoder_decoder'
    cap = min(config.max_new_tokens, model.config.max_context-1 if seq2seq else model.config.max_context-length)
    if cap < 1:
        raise ValueError('No context capacity for continuation')
    source = torch.tensor([item.ids for item in encoded], dtype=torch.long, device=device)
    memory = None
    with amp_context(device, dtype):
        if seq2seq:
            memory = model.encode(source)
            logits, cache = model(torch.full((len(prompts), 1), tokenizer.token_to_id('<bos>'),
                                             dtype=torch.long, device=device), memory=memory, use_cache=True)
        else:
            logits, cache = model(source, use_cache=True)
    forbidden = [tokenizer.token_to_id(t) for t in ['<pad>', '<bos>', '<eos>', '<unk>']]
    forbidden = [t for t in forbidden if t is not None]
    generators = [torch.Generator(device=device).manual_seed(int(s) % (2**63-1))
                  for s in (seeds or [config.seed]*len(prompts))]
    generated = [[] for _ in prompts]
    results = [None]*len(prompts)
    progress = [(0, 0) for _ in prompts]
    for index in range(cap):
        next_tokens = []
        for row in range(len(prompts)):
            if results[row] is not None:
                next_tokens.append(tokenizer.token_to_id('<bos>'))
                continue
            token = select_token(logits[row, -1], generated[row], forbidden, config, generators[row])
            next_tokens.append(token)
            generated[row].append(token)
            text = tokenizer.decode(generated[row], skip_special_tokens=True)
            words = text.split()
            if len(words) > config.words or (len(words) == config.words and text[-1:].isspace()):
                results[row] = (' '.join(words[:config.words]),
                    {'generated_tokens': len(generated[row]), 'prompt_tokens': length, 'cap_tokens': cap})
            else:
                count, last_progress = progress[row]
                if len(words) > count:
                    progress[row] = (len(words), index)
                elif index-last_progress >= config.stall_tokens:
                    results[row] = ShortGenerationError(f'Decoder stalled at {len(words)} words')
        if all(result is not None for result in results):
            break
        if index+1 < cap:
            with amp_context(device, dtype):
                logits, cache = model(torch.tensor(next_tokens, device=device)[:, None], memory=memory,
                                      cache=cache, use_cache=True)
    for row in range(len(prompts)):
        if results[row] is None:
            results[row] = ShortGenerationError(f'Generated only {len(tokenizer.decode(generated[row]).split())} words '
                f'without a boundary after word {config.words} in {cap} tokens')
    return results


def continuation(model, tokenizer, prompt, config=None, device=None, dtype=None):
    config = config or DecodingConfig()
    result = continuation_batch(model, tokenizer, [prompt], config,
                                device or next(model.parameters()).device, dtype)[0]
    if isinstance(result, Exception):
        raise result
    return result


def retry_short(model, tokenizer, prompt, config, device, dtype, seed, initial_error):
    errors = [str(initial_error)]
    retry = replace(config, temperature=0.8, top_p=0.9, repetition_penalty=max(1.2, config.repetition_penalty))
    for attempt in range(1, 3):
        result = continuation_batch(model, tokenizer, [prompt], retry, device, dtype, [seed+attempt])[0]
        if not isinstance(result, ShortGenerationError):
            text, info = result
            info.update(fallback=True, fallback_attempt=attempt, fallback_decoding=asdict(retry), prior_errors=errors)
            return text, info
        errors.append(str(result))
    raise ShortGenerationError('All decoding attempts were short: ' + '; '.join(errors))


def load_checkpoint(checkpoint_path, tokenizer_path=None, device=None):
    device = device or torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    checkpoint_path = Path(checkpoint_path)
    tokenizer_path = Path(tokenizer_path or checkpoint_path.parent / 'tokenizer.json')
    state = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    if tokenizer.get_vocab_size() != state['resolved']['model']['vocab_size']:
        raise ValueError('Checkpoint and tokenizer vocabulary sizes differ')
    expected = state['resolved'].get('tokenizer_sha256')
    if expected and file_hash(tokenizer_path) != expected:
        raise ValueError('Checkpoint and tokenizer identities differ')
    model = build_model(state['resolved']['model'])
    model.load_state_dict(state['model'])
    model.to(device).eval()
    for key in ['model', 'optimizer', 'scaler']:
        state.pop(key, None)
    return model, tokenizer, state


def generate(checkpoint_path, test_path, output_dir, config=None, tokenizer_path=None,
             prompt_words=512, precision_mode='auto', resume=False, batch_size=4,
             benchmark_examples=None):
    config = config or DecodingConfig()
    config.validate()
    if config.words != 128 or batch_size < 1:
        raise ValueError('Submissions require 128 words and a positive batch size')
    full_test = load_test(test_path, prompt_words)
    if benchmark_examples is not None and benchmark_examples < 1:
        raise ValueError('Benchmark example count must be positive')
    test = dict(list(full_test.items())[:benchmark_examples]) if benchmark_examples else full_test
    start = time.perf_counter()
    model, tokenizer, state = load_checkpoint(checkpoint_path, tokenizer_path)
    device = next(model.parameters()).device
    precision_name, dtype = precision(device, precision_mode)
    output = Path(output_dir)
    identity = {'checkpoint_sha256': file_hash(checkpoint_path), 'test_sha256': file_hash(test_path),
                'tokenizer_sha256': file_hash(tokenizer_path or Path(checkpoint_path).parent / 'tokenizer.json'),
                'decoding': asdict(config), 'precision': precision_name,
                'batch_size': batch_size, 'benchmark_examples': benchmark_examples}
    predictions, diagnostics = {}, {}
    if output.exists() and any(output.iterdir()):
        if not resume:
            raise FileExistsError(f'Generation directory exists: {output}; use --resume or a new path')
        if read_json(output / 'generation_identity.json') != identity:
            raise ValueError('Generation checkpoint, tokenizer, test or decoding changed')
        if (output / 'partial_predictions.json').exists():
            predictions = read_json(output / 'partial_predictions.json')
            diagnostics = read_json(output / 'generation_diagnostics.json')
            if not set(predictions) <= set(test):
                raise ValueError('Unknown IDs in partial predictions')
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / 'generation_identity.json', identity)
    buckets = defaultdict(list)
    pending = [(i, r) for i, r in test.items() if i not in predictions]
    for (identifier, record), encoded in zip(pending, tokenizer.encode_batch([r['text'].rstrip() for _, r in pending])):
        buckets[len(encoded.ids)].append((identifier, record))
    def save():
        write_json(output / 'partial_predictions.json', predictions)
        write_json(output / 'generation_diagnostics.json', diagnostics)
    processed = 0
    decode_start = time.perf_counter()
    for bucket in buckets.values():
        for offset in range(0, len(bucket), batch_size):
            items = bucket[offset:offset+batch_size]
            seeds = [config.seed+int(fingerprint(i)[:8], 16) for i, _ in items]
            try:
                results = continuation_batch(model, tokenizer, [r['text'] for _, r in items], config, device, dtype, seeds)
                for (identifier, _), result in zip(items, results):
                    if not isinstance(result, ShortGenerationError):
                        text, info = result
                        predictions[identifier], diagnostics[identifier] = {'text': text}, info
                save()
                for (identifier, record), result, id_seed in zip(items, results, seeds):
                    if isinstance(result, ShortGenerationError):
                        try:
                            text, info = retry_short(model, tokenizer, record['text'], config, device, dtype, id_seed, result)
                            predictions[identifier], diagnostics[identifier] = {'text': text}, info
                        except ShortGenerationError as error:
                            if not benchmark_examples:
                                raise
                            diagnostics[identifier] = {'short_output_error': str(error)}
                        save()
            except Exception as error:
                save()
                write_json(output / 'generation_error.json', {'ids': [i for i, _ in items], 'error': str(error)})
                raise
            processed += len(items)
            print(f'Generated {len(predictions)}/{len(test)}; session {time.perf_counter()-start:.1f}s', flush=True)
    archive = None
    if benchmark_examples:
        write_json(output / 'benchmark_predictions.json', predictions)
    else:
        archive = package(predictions, test, output)
    decode_seconds = time.perf_counter()-decode_start
    write_json(output / 'generation_summary.json', {'checkpoint_step': state['step'],
                'run_id': state['resolved']['run_id'], 'session_seconds': time.perf_counter()-start,
                'decode_seconds': decode_seconds, 'session_examples': processed,
                'examples': len(predictions), 'benchmark': bool(benchmark_examples),
                'projected_full_decode_seconds': decode_seconds/max(processed, 1)*len(full_test),
                'fallback_examples': sum(bool(d.get('fallback')) for d in diagnostics.values()),
                'short_examples': sum('short_output_error' in d for d in diagnostics.values()),
                'archive': str(archive.resolve()) if archive else None})
    return predictions


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ['checkpoint', 'test', 'output']:
        parser.add_argument('--'+name, required=True)
    parser.add_argument('--tokenizer')
    parser.add_argument('--prompt-words', type=int, default=512)
    parser.add_argument('--precision', default='auto', choices=['auto', 'fp32', 'bf16', 'fp16'])
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--batch-size', type=int, default=4)
    parser.add_argument('--benchmark-examples', type=int)
    for key in ['temperature', 'top_p', 'top_k', 'repetition_penalty', 'max_new_tokens', 'seed',
                'no_repeat_ngram_size', 'stall_tokens']:
        default = getattr(DecodingConfig(), key)
        parser.add_argument('--'+key.replace('_', '-'), type=type(default), default=default)
    args = vars(parser.parse_args())
    keys = set(asdict(DecodingConfig()))-{'words'}
    config = DecodingConfig(**{key: args.pop(key) for key in keys})
    generate(args.pop('checkpoint'), args.pop('test'), args.pop('output'), config,
             args.pop('tokenizer'), args.pop('prompt_words'), args.pop('precision'), **args)


if __name__ == '__main__':
    main()


### `phase1/models.py`


In [ ]:
%%writefile phase1/models.py
"""Four genuinely different randomly initialized language-model designs.

All classes expose token representations through `hidden` and `encode`, so
later phases can add task heads. Generation uses attention KV/recurrent caches.
The local attention implementation computes query chunks against a bounded
key window: it does not construct a quadratic full-length local mask.
"""
from dataclasses import dataclass
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "dense_gpt"
    vocab_size: int = 24000
    dim: int = 768
    layers: int = 12
    heads: int = 12
    kv_heads: int = 2
    ff_dim: int = 2048
    max_context: int = 2048
    encoder_layers: int = 6
    decoder_layers: int = 6
    local_window: int = 256
    query_chunk: int = 64
    global_every: int = 4
    dropout: float = 0.0
    gradient_checkpointing: bool = True

    def validate(self):
        if self.dim % self.heads or self.heads % self.kv_heads:
            raise ValueError("dim must be divisible by heads; heads must be divisible by kv_heads")
        if (self.dim // self.heads) % 2:
            raise ValueError("RoPE head dimension must be even")
        if min(self.layers, self.encoder_layers, self.decoder_layers, self.local_window,
               self.query_chunk, self.global_every, self.max_context) <= 0:
            raise ValueError("All layer/context/window counts must be positive")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must lie in [0,1)")


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        y = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)
        return y.to(x.dtype) * self.weight


def rotary(x, offset=0):
    dim = x.size(-1)
    inv = 1.0 / (10000 ** (torch.arange(0, dim, 2, device=x.device).float() / dim))
    angles = torch.outer(torch.arange(offset, offset + x.size(-2), device=x.device).float(), inv)
    cos, sin = angles.cos().to(x.dtype), angles.sin().to(x.dtype)
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, config, kv_heads=None, rope=False, local=False):
        super().__init__()
        self.heads = config.heads
        self.kv_heads = kv_heads or config.heads
        self.head_dim = config.dim // config.heads
        self.rope, self.local = rope, local
        self.window, self.chunk = config.local_window, config.query_chunk
        self.dropout = config.dropout
        self.q = nn.Linear(config.dim, config.dim, bias=False)
        self.k = nn.Linear(config.dim, self.kv_heads * self.head_dim, bias=False)
        self.v = nn.Linear(config.dim, self.kv_heads * self.head_dim, bias=False)
        self.o = nn.Linear(config.dim, config.dim, bias=False)

    def forward(self, x, context=None, past=None, causal=True, use_cache=False):
        batch, length, dim = x.shape
        q = self.q(x).view(batch, length, self.heads, self.head_dim).transpose(1, 2)
        cross = context is not None
        offset = past[0].size(-2) if past is not None and not cross else 0
        if cross and past is not None:
            k, v = past
        else:
            source = x if context is None else context
            k = self.k(source).view(batch, -1, self.kv_heads, self.head_dim).transpose(1, 2)
            v = self.v(source).view(batch, -1, self.kv_heads, self.head_dim).transpose(1, 2)
            if self.rope and not cross:
                k = rotary(k, offset)
            if past is not None and not cross:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rope and not cross:
            q = rotary(q, offset)
        kwargs = dict(dropout_p=self.dropout if self.training else 0.0,
                      enable_gqa=self.kv_heads != self.heads)
        if self.local and causal:
            parts = []
            for start in range(0, length, self.chunk):
                end = min(start + self.chunk, length)
                left, right = max(0, offset + start - self.window + 1), offset + end
                qi = torch.arange(offset + start, offset + end, device=x.device)[:, None]
                ki = torch.arange(left, right, device=x.device)[None, :]
                allowed = (ki <= qi) & (ki > qi - self.window)
                parts.append(F.scaled_dot_product_attention(
                    q[:, :, start:end], k[:, :, left:right], v[:, :, left:right],
                    attn_mask=allowed, **kwargs))
            y = torch.cat(parts, -2)
        elif causal and past is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, **kwargs)
        elif causal and length > 1:
            qi = torch.arange(offset, offset + length, device=x.device)[:, None]
            ki = torch.arange(k.size(-2), device=x.device)[None, :]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=ki <= qi, **kwargs)
        else:
            y = F.scaled_dot_product_attention(q, k, v, **kwargs)
        y = y.transpose(1, 2).contiguous().view(batch, length, dim)
        return self.o(y), (k, v) if use_cache else None


class FeedForward(nn.Module):
    def __init__(self, config, gated=True):
        super().__init__()
        self.gated = gated
        middle = config.ff_dim if gated else 4 * config.dim
        self.up = nn.Linear(config.dim, middle, bias=False)
        self.gate = nn.Linear(config.dim, middle, bias=False) if gated else None
        self.down = nn.Linear(middle, config.dim, bias=False)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        y = F.silu(self.gate(x)) * self.up(x) if self.gated else F.gelu(self.up(x))
        return self.drop(self.down(y))


class Block(nn.Module):
    def __init__(self, config, kind="dense", local=False, cross=False):
        super().__init__()
        dense = kind == "dense"
        norm = nn.LayerNorm if dense else RMSNorm
        self.norm1, self.norm2 = norm(config.dim), norm(config.dim)
        self.attn = Attention(config, config.kv_heads if kind == "gqa" else config.heads,
                              rope=kind in {"gqa", "hybrid"}, local=local)
        self.ff = FeedForward(config, gated=not dense)
        self.cross_attn = Attention(config) if cross else None
        self.cross_norm = norm(config.dim) if cross else None
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x, memory=None, past=None, causal=True, use_cache=False):
        past = past or {}
        y, self_cache = self.attn(self.norm1(x), past=past.get("self"),
                                  causal=causal, use_cache=use_cache)
        x = x + self.drop(y)
        cross_cache = None
        if self.cross_attn is not None:
            y, cross_cache = self.cross_attn(self.cross_norm(x), context=memory,
                                           past=past.get("cross"), causal=False,
                                           use_cache=use_cache)
            x = x + self.drop(y)
        x = x + self.ff(self.norm2(x))
        return x, {"self": self_cache, "cross": cross_cache} if use_cache else None


class RecurrentBlock(nn.Module):
    """LSTM state propagation replaces attention in alternating hybrid layers."""
    def __init__(self, config):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(config.dim), RMSNorm(config.dim)
        self.rnn = nn.LSTM(config.dim, config.dim, batch_first=True)
        self.ff = FeedForward(config)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x, memory=None, past=None, causal=True, use_cache=False):
        y, state = self.rnn(self.norm1(x), past.get("state") if past else None)
        x = x + self.drop(y)
        x = x + self.ff(self.norm2(x))
        return x, {"state": state} if use_cache else None


def initialize(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)


class DecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.dim)
        self.position = nn.Embedding(config.max_context, config.dim) if config.architecture == "dense_gpt" else None
        blocks = []
        for layer in range(config.layers):
            if config.architecture == "recurrent_hybrid" and layer % 2 == 0:
                blocks.append(RecurrentBlock(config))
            else:
                kind = {"dense_gpt": "dense", "gqa_local": "gqa", "recurrent_hybrid": "hybrid"}[config.architecture]
                local = kind == "gqa" and (layer + 1) % config.global_every != 0
                blocks.append(Block(config, kind=kind, local=local))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(config.dim) if config.architecture == "dense_gpt" else RMSNorm(config.dim)
        self.apply(initialize)
        if config.architecture == "dense_gpt":
            residual_std = 0.02 / math.sqrt(2 * config.layers)
            for block in self.blocks:
                nn.init.normal_(block.attn.o.weight, std=residual_std)
                nn.init.normal_(block.ff.down.weight, std=residual_std)

    def hidden(self, input_ids, cache=None, use_cache=False):
        offset = cache["length"] if cache else 0
        if input_ids.size(1) + offset > self.config.max_context:
            raise ValueError("Input plus cache exceeds max_context; no silent prompt truncation")
        x = self.embedding(input_ids)
        if self.position is not None:
            x = x + self.position(torch.arange(offset, offset + input_ids.size(1), device=x.device))
        new = []
        for i, block in enumerate(self.blocks):
            past = cache["layers"][i] if cache else None
            if self.training and self.config.gradient_checkpointing and not use_cache:
                x = checkpoint(lambda y, b=block: b(y)[0], x, use_reentrant=False)
                state = None
            else:
                x, state = block(x, past=past, use_cache=use_cache)
            new.append(state)
        state = {"length": offset + input_ids.size(1), "layers": new} if use_cache else None
        return self.norm(x), state

    def forward(self, input_ids, source_ids=None, memory=None, cache=None, use_cache=False):
        x, cache = self.hidden(input_ids, cache, use_cache)
        return F.linear(x, self.embedding.weight), cache


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.dim)
        self.position = nn.Embedding(config.max_context, config.dim)
        self.encoder = nn.ModuleList(Block(config, kind="seq2seq") for _ in range(config.encoder_layers))
        self.decoder = nn.ModuleList(Block(config, kind="seq2seq", cross=True) for _ in range(config.decoder_layers))
        self.encoder_norm, self.decoder_norm = RMSNorm(config.dim), RMSNorm(config.dim)
        self.apply(initialize)

    def encode(self, source_ids):
        if source_ids.size(1) > self.config.max_context:
            raise ValueError("Encoder input exceeds max_context")
        x = self.embedding(source_ids) + self.position(torch.arange(source_ids.size(1), device=source_ids.device))
        for block in self.encoder:
            if self.training and self.config.gradient_checkpointing:
                x = checkpoint(lambda y, b=block: b(y, causal=False)[0], x, use_reentrant=False)
            else:
                x, _ = block(x, causal=False)
        return self.encoder_norm(x)

    def hidden(self, input_ids, memory, cache=None, use_cache=False):
        offset = cache["length"] if cache else 0
        if offset + input_ids.size(1) > self.config.max_context:
            raise ValueError("Decoder input plus cache exceeds max_context")
        x = self.embedding(input_ids) + self.position(torch.arange(offset, offset + input_ids.size(1), device=input_ids.device))
        new = []
        for i, block in enumerate(self.decoder):
            if self.training and self.config.gradient_checkpointing and not use_cache:
                x = checkpoint(lambda y, m, b=block: b(y, memory=m)[0], x, memory, use_reentrant=False)
                state = None
            else:
                x, state = block(x, memory=memory, past=cache["layers"][i] if cache else None, use_cache=use_cache)
            new.append(state)
        state = {"length": offset + input_ids.size(1), "layers": new} if use_cache else None
        return self.decoder_norm(x), state

    def forward(self, input_ids, source_ids=None, memory=None, cache=None, use_cache=False):
        if memory is None:
            if source_ids is None:
                raise ValueError("Encoder-decoder requires source_ids or encoder memory")
            memory = self.encode(source_ids)
        x, cache = self.hidden(input_ids, memory, cache, use_cache)
        return F.linear(x, self.embedding.weight), cache


def build_model(config):
    if isinstance(config, dict):
        config = ModelConfig(**config)
    config.validate()
    if config.architecture == "encoder_decoder":
        return EncoderDecoderLM(config)
    if config.architecture in {"dense_gpt", "gqa_local", "recurrent_hybrid"}:
        return DecoderLM(config)
    raise ValueError(f"Unknown architecture: {config.architecture}")


def parameter_count(config):
    # Instantiate on meta: count real tied parameters without allocating 100M weights.
    with torch.device("meta"):
        model = build_model(config)
    return sum(p.numel() for p in model.parameters())


### `phase1/official_eval.py`


In [ ]:
%%writefile phase1/official_eval.py
"""Execute the supplied Phase 1 scorer unchanged and retain its exact output."""
import argparse
import math
from pathlib import Path
import re
import subprocess
import sys
from .common import file_hash, read_json, write_json

DEFAULT_SCORER = Path(__file__).resolve().parent.parent / 'evaluators' / 'evaluate_phase1.py'


def parse_scores(stdout):
    scores = {}
    for label, key in [('BERTScore F1', 'official_bertscore'), ('ROUGE-L F1', 'official_rouge_l'), ('BLEU-4', 'official_bleu4')]:
        match = re.search(r'^'+re.escape(label)+r':\s*([-+\deE.]+)\s*$', stdout, re.M)
        if not match or not math.isfinite(float(match.group(1))):
            raise ValueError(f'Official scorer did not emit a finite {label}')
        scores[key] = float(match.group(1))
    return scores


def score(prediction, reference, output, scorer=DEFAULT_SCORER):
    scorer = Path(scorer).resolve()
    argv = [sys.executable, str(scorer), '--truth', str(reference), '--prediction', str(prediction)]
    result = subprocess.run(argv, capture_output=True, text=True)
    evidence = {'argv': argv, 'scorer_sha256': file_hash(scorer),
                'prediction_sha256': file_hash(prediction), 'reference_sha256': file_hash(reference),
                'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}
    write_json(output, evidence)
    if result.returncode:
        raise RuntimeError(f'Official evaluator exited {result.returncode}; see {output}: {result.stderr[-2000:]}')
    evidence.update(parse_scores(result.stdout))
    write_json(output, evidence)
    return evidence


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    for key in ['prediction', 'reference', 'output']:
        ap.add_argument('--'+key, required=True)
    ap.add_argument('--scorer', default=str(DEFAULT_SCORER))
    a = ap.parse_args()
    print(score(a.prediction, a.reference, a.output, a.scorer))


if __name__ == '__main__':
    main()


### `phase1/profile.py`


In [ ]:
%%writefile phase1/profile.py
"""Measure prompt lengths and throughput; derive budgets and disk projections."""
import argparse
import math
import shutil
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer
from .common import directory_bytes, read_json, write_json
from .submission import load_test


def prompt_profile(test_path, tokenizer_path, reserve_tokens=250, max_context=2048):
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    test = load_test(test_path)
    lengths = np.array([len(e.ids) for e in tokenizer.encode_batch([r['text'].rstrip() for r in test.values()])])
    p99, maximum = int(np.quantile(lengths, .99, method='higher')), int(lengths.max())
    if maximum+reserve_tokens > max_context:
        raise ValueError(f'Longest prompt {maximum} + reserve {reserve_tokens} exceeds context {max_context}; revise tokenizer/context before training')
    sequence = 1024 if p99+reserve_tokens <= 1024 else max_context
    return {'examples': len(test), 'p50': int(np.median(lengths)), 'p99': p99, 'max': maximum,
            'reserve_tokens': reserve_tokens, 'sequence_length': sequence,
            # Learned absolute positions need exposure beyond the short sequence.
            'long_sequence_length': max_context if sequence < max_context else None,
            'long_sequence_probability': 0.1 if sequence < max_context else 0.0,
            'source_length_range': [max(1, int(np.quantile(lengths, .05))), int(np.quantile(lengths, .95))]}


def measured_budget(run_dir, hours, safety=0.85):
    if hours <= 0 or not 0 < safety <= 1:
        raise ValueError('Hours must be positive and safety in (0,1]')
    run_dir = Path(run_dir)
    status = read_json(run_dir/'status.json')
    config = read_json(run_dir/'resolved_config.json')
    if status['step'] < 50:
        raise ValueError('Measure at least 50 optimizer steps before budgeting')
    wall_rate = status['input_tokens_seen']/status.get('wall_seconds', status['session_seconds'])
    train_rate = status['input_tokens_seen']/status['train_seconds']
    tokens_per_step = config['tokens_per_step']
    budget = max(tokens_per_step, int(min(wall_rate, train_rate)*hours*3600*safety)//tokens_per_step*tokens_per_step)
    return {'token_budget': budget, 'wall_tokens_per_second': wall_rate,
            'train_tokens_per_second': train_rate, 'hours': hours, 'safety': safety,
            'profile_step': status['step'], 'training_targets_per_input': status['target_tokens_seen']/status['input_tokens_seen']}


def disk_check(project, parameters, train_tokens, val_tokens, active_jobs=2, limit_bytes=None, token_bytes=2):
    project = Path(project)
    # Existing pilot logs/models plus final best/latest model-only + full Adam
    # checkpoints. Two simultaneous atomic full-checkpoint writes need headroom.
    additional = sum(p*20 for p in parameters)+sum(sorted(parameters, reverse=True)[:active_jobs])*12
    if token_bytes not in {2, 4} or active_jobs < 1:
        raise ValueError('Invalid token storage dtype or active job count')
    additional += (train_tokens+val_tokens)*token_bytes + 512*1024**2
    current = directory_bytes(project)
    free = shutil.disk_usage(project).free
    result = {'current_saved_bytes': current, 'additional_peak_estimate_bytes': additional,
              'projected_saved_peak_bytes': current+additional, 'saved_limit_bytes': limit_bytes,
              'filesystem_free_bytes': free, 'active_jobs': active_jobs}
    if (limit_bytes is not None and current+additional > limit_bytes) or additional > free:
        raise OSError(f'Insufficient projected disk headroom: {result}; reduce corpus/budgets or retained checkpoints')
    return result


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--test', required=True)
    ap.add_argument('--tokenizer', required=True)
    ap.add_argument('--output', required=True)
    a = ap.parse_args()
    result = prompt_profile(a.test, a.tokenizer)
    write_json(a.output, result)
    print(result)


if __name__ == '__main__':
    main()


### `phase1/runtime.py`


In [ ]:
%%writefile phase1/runtime.py
"""Hardware validation and batch sizing for the two-GPU Jupyter workflow."""
from pathlib import Path
from .scheduler import visible_gpu_ids


def gpu_preflight(count=2, minimum_vram_gib=22):
    import torch
    from .train import precision
    if not torch.cuda.is_available() or torch.cuda.device_count() < count:
        raise RuntimeError(f'This workflow requires {count} visible CUDA GPUs. Check the notebook kernel and CUDA_VISIBLE_DEVICES.')
    visible = visible_gpu_ids()
    if len(visible) < count:
        raise RuntimeError('GPU visibility mapping does not contain enough usable devices')
    rows = []
    for logical in range(count):
        device = torch.device(f'cuda:{logical}')
        props = torch.cuda.get_device_properties(device)
        total = props.total_memory / 1024**3
        if total < minimum_vram_gib:
            raise RuntimeError(f'GPU {logical} has {total:.1f} GiB; this configuration expects about 24 GB per GPU')
        mode, _ = precision(device)
        free, _ = torch.cuda.mem_get_info(device)
        rows.append({'logical_device': logical, 'worker_gpu_id': visible[logical],
                     'name': props.name, 'total_vram_gib': round(total, 2),
                     'free_vram_gib': round(free/1024**3, 2), 'precision': mode})
    return rows


def accumulation(sequence_length, micro_batch_size=2, tokens_per_update=32768,
                 long_sequence_length=None):
    if min(sequence_length, micro_batch_size, tokens_per_update) < 1:
        raise ValueError('Batch dimensions must be positive')
    for length in [sequence_length, long_sequence_length]:
        if length is not None and tokens_per_update % (length*micro_batch_size):
            raise ValueError('Effective token count must divide both short and long micro-batches')
    return tokens_per_update // (sequence_length*micro_batch_size)


### `phase1/scheduler.py`


In [ ]:
%%writefile phase1/scheduler.py
"""One isolated child per GPU, reusable for training, profiling and inference."""
import argparse
import os
from pathlib import Path
import subprocess
import time
from .common import read_json, write_json


def visible_gpu_ids():
    import torch
    visible = os.environ.get('CUDA_VISIBLE_DEVICES')
    if visible is not None:
        return [x.strip() for x in visible.split(',') if x.strip() and x.strip() != '-1']
    return [str(i) for i in range(torch.cuda.device_count())]


def run_jobs(jobs, gpus=None, cwd=None, status_path=None):
    """Keep completed work/logs on failure; each child sees its own cuda:0.

    GPU identifiers are physical IDs (or UUIDs), including the inherited visible
    mapping. An empty GPU list runs a single CPU worker. No shell is invoked.
    """
    gpus = visible_gpu_ids() if gpus is None else [str(g) for g in gpus]
    if len(set(gpus)) != len(gpus):
        raise ValueError('GPU assignments must be unique')
    if len({job['id'] for job in jobs}) != len(jobs):
        raise ValueError('Job IDs must be unique')
    for job in jobs:
        if not job.get('argv') or any(not isinstance(x, str) for x in job['argv']):
            raise ValueError('Each job needs a nonempty argv list of strings')
    pending, active, rows = list(jobs), {}, {}
    slots = gpus or ['']
    last_notice = time.monotonic()
    def persist():
        if status_path:
            write_json(status_path, rows)
    try:
        while pending or active:
            for gpu in slots:
                if pending and gpu not in active:
                    job = pending.pop(0)
                    log_path = Path(job['log'])
                    log_path.parent.mkdir(parents=True, exist_ok=True)
                    log = log_path.open('a', encoding='utf-8')
                    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED='1', OMP_NUM_THREADS='2')
                    try:
                        process = subprocess.Popen(job['argv'], cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
                    except Exception:
                        log.close()
                        raise
                    active[gpu] = (process, job, log, time.monotonic())
                    rows[job['id']] = {'status': 'running', 'gpu': gpu or 'CPU', 'log': str(log_path), 'argv': job['argv']}
                    print(f"START {job['id']} on {gpu or 'CPU'}; log {log_path}", flush=True)
                    persist()
            for gpu, (process, job, log, tick) in list(active.items()):
                code = process.poll()
                if code is not None:
                    log.close()
                    rows[job['id']].update(status='completed' if code == 0 else 'failed',
                                           returncode=code, seconds=time.monotonic()-tick)
                    del active[gpu]
                    print(f"END {job['id']}: returncode={code}", flush=True)
                    persist()
            if time.monotonic()-last_notice >= 30:
                print('Running:', [job['id'] for _, job, _, _ in active.values()], 'queued:', len(pending), flush=True)
                last_notice = time.monotonic()
            if pending or active:
                time.sleep(0.2)
    except BaseException:
        for process, job, log, _ in active.values():
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            log.close()
            rows[job['id']]['status'] = 'interrupted'
        persist()
        raise
    failed = [name for name, row in rows.items() if row['status'] != 'completed']
    if failed:
        raise RuntimeError(f'Jobs failed: {failed}; inspect per-job logs; successful outputs are preserved')
    return rows


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--jobs-json', required=True)
    ap.add_argument('--gpus', nargs='*')
    ap.add_argument('--cwd')
    ap.add_argument('--status', required=True)
    a = ap.parse_args()
    run_jobs(read_json(a.jobs_json), a.gpus, a.cwd, a.status)


if __name__ == '__main__':
    main()


### `phase1/select.py`


In [ ]:
%%writefile phase1/select.py
"""Select a checkpoint/decoder with official held-out BERTScore, retaining evidence."""
import argparse
from dataclasses import asdict
from pathlib import Path
from .common import file_hash, fingerprint, read_json, write_json
from .generate import DecodingConfig, generate
from .official_eval import DEFAULT_SCORER, score

DECODERS = [
    DecodingConfig(temperature=0.0, repetition_penalty=1.15),
    DecodingConfig(temperature=0.1, top_p=0.95, repetition_penalty=1.1),
    DecodingConfig(temperature=0.7, top_p=0.9, repetition_penalty=1.2, no_repeat_ngram_size=3),
]


def select(run_dir, examples_path, scorer=DEFAULT_SCORER, batch_size=4, screen_examples=16):
    run_dir = Path(run_dir)
    examples = read_json(examples_path)
    if not examples or any(not r.get('reference') for r in examples.values()):
        raise ValueError('Selection requires held-out prompts and references')
    # Balanced screening across sources. Every architecture uses this same order.
    groups = {}
    for identifier, record in sorted(examples.items()):
        groups.setdefault(str(record.get('source_id', 0)), []).append((identifier, record))
    ordered = []
    while any(groups.values()):
        for group in groups.values():
            if group:
                ordered.append(group.pop(0))
    if screen_examples < 1:
        raise ValueError('Screen example count must be positive')
    identity = {'checkpoints': {name:file_hash(run_dir/name) for name in ['best.pt','latest.pt']
                               if (run_dir/name).exists()},
                'examples':file_hash(examples_path), 'scorer':file_hash(scorer),
                'decoders':[asdict(d) for d in DECODERS], 'screen_examples':screen_examples,
                'batch_size':batch_size}
    selection_path = run_dir/'selection.json'
    if selection_path.exists() and read_json(selection_path).get('identity') == identity:
        return read_json(selection_path)
    root = run_dir / 'selection' / fingerprint(identity)[:20]
    root.mkdir(parents=True, exist_ok=True)
    screen = dict(ordered[:min(screen_examples, len(examples))])
    write_json(root/'screen_prompts.json', screen)
    write_json(root/'screen_truth.json', {i: {'text': r['reference']} for i, r in screen.items()})
    write_json(root/'full_truth.json', {i: {'text': r['reference']} for i, r in examples.items()})
    checkpoints, steps = [], set()
    import torch
    for name in ['best.pt', 'latest.pt']:
        path = run_dir/name
        if path.exists():
            state = torch.load(path, map_location='cpu', weights_only=False)
            step = state['step']
            del state
            if step not in steps:
                checkpoints.append(path)
                steps.add(step)
    if not checkpoints:
        raise FileNotFoundError('No best/latest model-only checkpoint')
    candidates, errors = [], []
    for checkpoint in checkpoints:
        for index, decoder in enumerate(DECODERS):
            output = root/f'{checkpoint.stem}_decoder{index}'
            try:
                generate(checkpoint, root/'screen_prompts.json', output, decoder,
                         resume=output.exists(), batch_size=batch_size)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                evidence = score(output/'prediction.json', root/'screen_truth.json', output/'official_metrics.json', scorer)
                row = {'checkpoint': str(checkpoint.resolve()), 'checkpoint_sha256': file_hash(checkpoint),
                       'decoding': asdict(decoder), 'official_bertscore': evidence['official_bertscore'],
                       'evidence': str((output/'official_metrics.json').resolve())}
                candidates.append(row)
            except Exception as error:
                errors.append({'checkpoint': str(checkpoint), 'decoder': index, 'error': str(error)})
            write_json(root/'candidates.json', {'scores': candidates, 'errors': errors})
    if not candidates:
        raise RuntimeError(f'All selection candidates failed; inspect {root}/candidates.json')
    # Screening is the controlled comparison. Confirm its winner on the full set;
    # do not claim all candidates received a full-set evaluation.
    winner = max(candidates, key=lambda row: row['official_bertscore'])
    full_dir = root/'full_winner'
    decoder = DecodingConfig(**winner['decoding'])
    generate(winner['checkpoint'], examples_path, full_dir, decoder,
             resume=full_dir.exists(), batch_size=batch_size)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    evidence = score(full_dir/'prediction.json', root/'full_truth.json', full_dir/'official_metrics.json', scorer)
    selection = dict(winner, identity=identity, screen_bertscore=winner['official_bertscore'],
                     official_bertscore=evidence['official_bertscore'],
                     selection_examples=len(screen), confirmation_examples=len(examples),
                     scorer_sha256=file_hash(scorer), examples_sha256=file_hash(examples_path),
                     evidence=str((full_dir/'official_metrics.json').resolve()),
                     criterion='highest official BERTScore on balanced screening set; winner confirmed on full held-out set')
    write_json(run_dir/'selection.json', selection)
    return selection


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--run', required=True)
    ap.add_argument('--examples', required=True)
    ap.add_argument('--scorer', default=str(DEFAULT_SCORER))
    ap.add_argument('--batch-size', type=int, default=4)
    ap.add_argument('--screen-examples', type=int, default=16)
    a = ap.parse_args()
    print(select(a.run, a.examples, a.scorer, a.batch_size, a.screen_examples))


if __name__ == '__main__':
    main()


### `phase1/submission.py`


In [ ]:
%%writefile phase1/submission.py
"""Strict Phase 1 schema, ID, word-count and ZIP checks (stdlib only)."""
import argparse
import json
import zipfile
import warnings
from pathlib import Path
from .common import read_json, unique_object, write_json


def load_test(path, expected_prompt_words=512, strict_prompt_words=False):
    data = read_json(path)
    if not isinstance(data, dict) or not data:
        raise ValueError("Expected a nonempty object mapping official IDs to {'text': prompt}")
    for identifier, record in data.items():
        if not isinstance(identifier, str) or not isinstance(record, dict) or not isinstance(record.get("text"), str):
            raise ValueError(f"Invalid test record: {identifier!r}")
        if not record['text'].strip():
            raise ValueError(f"Empty test prompt: {identifier!r}")
        if expected_prompt_words is not None and len(record["text"].split()) != expected_prompt_words:
            message = f"Test ID {identifier!r} has {len(record['text'].split())} whitespace words; expected {expected_prompt_words}. Full prompt is retained."
            if strict_prompt_words:
                raise ValueError(message)
            warnings.warn(message, RuntimeWarning)
    return data


def validate(predictions, test, words=128):
    if not isinstance(predictions, dict):
        raise ValueError("Phase 1 predictions must be a JSON object")
    if set(predictions) != set(test):
        raise ValueError(f"ID mismatch: missing={sorted(set(test)-set(predictions))[:10]}, unknown={sorted(set(predictions)-set(test))[:10]}")
    for identifier, record in predictions.items():
        if not isinstance(record, dict) or set(record) != {"text"} or not isinstance(record["text"], str):
            raise ValueError(f"Invalid prediction schema for ID {identifier}")
        if len(record["text"].split()) != words:
            raise ValueError(f"Prediction {identifier}: expected exactly {words} words, got {len(record['text'].split())}")
        if "\x00" in record["text"]:
            raise ValueError(f"NUL character in prediction {identifier}")
        prompt = test[identifier]["text"].strip()
        if prompt and record["text"].strip().startswith(prompt):
            raise ValueError(f"Prediction {identifier} contains the original prompt")
    return {"valid": True, "examples": len(predictions), "words_per_prediction": words}


def package(predictions, test, output_dir):
    result = validate(predictions, test)
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / "prediction.json", predictions)
    archive = output / "submission.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
        handle.write(output / "prediction.json", arcname="prediction.json")
    validate_archive(archive, test)
    write_json(output / "validation.json", result)
    return archive


def validate_archive(archive, test):
    with zipfile.ZipFile(archive) as handle:
        if handle.namelist() != ["prediction.json"]:
            raise ValueError("ZIP must contain exactly prediction.json at its root")
        data = json.loads(handle.read("prediction.json").decode("utf-8"), object_pairs_hook=unique_object)
    return validate(data, test)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test", required=True)
    parser.add_argument("--predictions")
    parser.add_argument("--zip")
    parser.add_argument("--prompt-words", type=int, default=512)
    args = parser.parse_args()
    if bool(args.predictions) == bool(args.zip):
        parser.error("Specify exactly one of --predictions or --zip")
    test = load_test(args.test, args.prompt_words)
    print(validate_archive(args.zip, test) if args.zip else validate(read_json(args.predictions), test))


if __name__ == "__main__":
    main()


### `phase1/train.py`


In [ ]:
%%writefile phase1/train.py
"""Bounded single-GPU training with mixed precision and exact-state resume.

Training samples blocks with replacement. `token_budget` counts tokens
processed, not a promise to cover every unique corpus token once.
"""
import argparse
from contextlib import nullcontext
from dataclasses import asdict
from datetime import datetime, timezone
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
import torch
from torch.nn import functional as F
from .common import file_hash, fingerprint, read_json, seed_everything, write_json, quarantine_directory
from .data import TokenCorpus
from .models import ModelConfig, build_model, parameter_count


def precision(device, requested="auto"):
    if device.type != "cuda":
        if requested not in {"auto", "fp32"}:
            raise ValueError("CPU smoke tests require fp32")
        return "fp32", None
    native_bf16 = torch.cuda.get_device_capability(device)[0] >= 8
    choice = ("bf16" if native_bf16 else "fp16") if requested == "auto" else requested
    if choice == "bf16" and not native_bf16:
        raise ValueError("bf16 is emulated on this GPU; use fp16")
    return choice, {"fp32": None, "bf16": torch.bfloat16, "fp16": torch.float16}[choice]


def amp_context(device, dtype):
    return torch.autocast(device_type=device.type, dtype=dtype) if dtype is not None else nullcontext()


def loss_for_block(model, tokens, bos_id, source_fraction=0.75, cut=None):
    if model.config.architecture == "encoder_decoder":
        cut = int(tokens.size(1) * source_fraction) if cut is None else cut
        if not 1 <= cut < tokens.size(1):
            raise ValueError("source_fraction must leave nonempty encoder and decoder sequences")
        source, labels = tokens[:, :cut], tokens[:, cut:]
        start = torch.full_like(labels[:, :1], bos_id)
        inputs = torch.cat((start, labels[:, :-1]), dim=1)
        logits, _ = model(inputs, source_ids=source)
    else:
        logits, _ = model(tokens[:, :-1])
        labels = tokens[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1))
    return loss, labels.numel()


def source_cut(settings, length, rng):
    span = settings.get("source_length_range")
    if span is None:
        fractions = settings.get("source_fraction_range")
        if fractions is None:
            return int(length * settings["source_fraction"])
        span = [int(length * fractions[0]), int(length * fractions[1])]
    low, high = max(1, int(span[0])), min(length-1, int(span[1]))
    if high < low:
        raise ValueError("Encoder source length range leaves no decoder targets")
    return int(rng.integers(low, high+1))


@torch.no_grad()
def evaluate(model, corpus, settings, bos_id, device, dtype):
    model.eval()
    rng = np.random.default_rng(settings["seed"] + 1000)
    weighted_loss, targets = 0.0, 0
    length = settings["sequence_length"] + (model.config.architecture != "encoder_decoder")
    for _ in range(settings["eval_batches"]):
        tokens = torch.from_numpy(corpus.sample(rng, settings["micro_batch_size"], length)).to(device)
        with amp_context(device, dtype):
            cut = source_cut(settings, length, rng) if model.config.architecture == "encoder_decoder" else None
            loss, count = loss_for_block(model, tokens, bos_id, settings["source_fraction"], cut)
        weighted_loss += loss.item() * count
        targets += count
    model.train()
    mean = weighted_loss / targets
    if not math.isfinite(mean):
        raise FloatingPointError("Nonfinite validation loss")
    return {"validation_loss": mean, "perplexity": math.exp(min(mean, 700)),
            "validation_target_tokens": targets}


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def environment():
    try:
        revision = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        revision = None
    return {"python": sys.version, "platform": platform.platform(), "git_revision": revision,
            "versions": {name: importlib.metadata.version(name) for name in ["torch", "numpy", "tokenizers", "ijson", "huggingface_hub"]},
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"), "command": sys.argv}


def source_digest(training_only=False):
    root = Path(__file__).parent
    names = {"train.py", "models.py", "data.py", "common.py"}
    return fingerprint({path.name: file_hash(path) for path in sorted(root.glob("*.py"))
                        if not training_only or path.name in names})


def check_resume(previous, current, allow_code_change=False):
    signature = previous.get("resume_signature")
    if signature is None:
        # Pre-patch checkpoints can be reviewed explicitly without pretending
        # they used the new source cuts/context settings.
        legacy = {key: previous[key] for key in
                  ["run_id", "model", "training", "data_fingerprint", "tokenizer_sha256"]}
        legacy["training"] = {"source_fraction_range": None, "source_length_range": None,
                              "long_sequence_length": None, "long_sequence_probability": 0.0,
                              "require_corpus_budget": False, **legacy["training"]}
        signature = fingerprint(legacy)
    if signature != current["resume_signature"]:
        raise ValueError("Resume model, training settings or data changed; use a new run_id/output path")
    changed = previous.get("training_code_fingerprint") != current["training_code_fingerprint"]
    if changed and not allow_code_change:
        raise ValueError("Training code changed; use a new run or explicitly --allow-code-change after review")
    return changed


def resolve(config_file, data_dir):
    config = read_json(config_file)
    metadata = read_json(Path(data_dir) / "metadata.json")
    model_config = dict(config["model"])
    model_config["vocab_size"] = metadata["vocab_size"]
    model_config = asdict(ModelConfig(**model_config))
    settings = dict(config["training"])
    defaults = {"seed": 42, "sequence_length": 1024, "micro_batch_size": 1,
                "gradient_accumulation": 16, "token_budget": 20_000_000,
                "learning_rate": 3e-4, "min_lr_ratio": 0.1, "warmup_fraction": 0.03,
                "weight_decay": 0.1, "grad_clip": 1.0, "eval_every": 250,
                "eval_batches": 16, "source_fraction": 0.75, "precision": "auto",
                "source_fraction_range": None, "source_length_range": None,
                "long_sequence_length": None, "long_sequence_probability": 0.0,
                "require_corpus_budget": False}
    unknown = set(settings) - set(defaults)
    if unknown:
        raise ValueError(f"Unsupported training settings: {sorted(unknown)}")
    settings = defaults | settings
    if min(settings[key] for key in ["sequence_length", "micro_batch_size", "gradient_accumulation", "token_budget", "eval_every", "eval_batches"]) <= 0:
        raise ValueError("Training budgets/batches/intervals must be positive")
    if not 0 < settings["source_fraction"] < 1 or not 0 <= settings["warmup_fraction"] < 1:
        raise ValueError("Invalid source/warmup fraction")
    if settings["sequence_length"] > model_config["max_context"]:
        raise ValueError("Training sequence exceeds model max_context")
    tokens_per_step = settings["sequence_length"] * settings["micro_batch_size"] * settings["gradient_accumulation"]
    if not 0 <= settings["long_sequence_probability"] <= 1:
        raise ValueError("Invalid long context probability")
    long_length = settings["long_sequence_length"]
    if long_length is not None and (not settings["sequence_length"] <= long_length <= model_config["max_context"]
                                    or tokens_per_step % (long_length * settings["micro_batch_size"])):
        raise ValueError("Long context must fit model and divide the effective tokens per update")
    for key in ["source_fraction_range", "source_length_range"]:
        span = settings[key]
        if span is not None and (len(span) != 2 or not 0 < span[0] <= span[1] or
                                 (key == "source_fraction_range" and span[1] >= 1)):
            raise ValueError(f"Invalid {key}")
    stored_train_tokens = metadata["counts"]["train"]
    if settings["require_corpus_budget"] and stored_train_tokens < settings["token_budget"]:
        raise ValueError("Prepared corpus is smaller than token budget; prepare more data or explicitly permit resampling")
    total_steps = math.ceil(settings["token_budget"] / tokens_per_step)
    resolved = {"run_id": config["run_id"], "model": model_config, "training": settings,
                "data_directory": str(Path(data_dir).resolve()), "data_fingerprint": fingerprint(metadata),
                "tokenizer_sha256": metadata["tokenizer_sha256"],
                "code_fingerprint": source_digest(), "total_steps": total_steps,
                "training_code_fingerprint": source_digest(True),
                "stored_train_tokens": stored_train_tokens,
                "processed_to_stored_token_ratio": total_steps*tokens_per_step / stored_train_tokens,
                "tokens_per_step": tokens_per_step, "parameters": parameter_count(model_config)}
    resolved["resume_signature"] = fingerprint({key: resolved[key] for key in
        ["run_id", "model", "training", "data_fingerprint", "tokenizer_sha256"]})
    resolved["run_fingerprint"] = fingerprint(resolved)
    return resolved


def train(config_file, data_dir, output_dir, resume=False, stop_after_step=None,
          max_hours=None, allow_small_model=False, dry_run=False, allow_code_change=False):
    resolved = resolve(config_file, data_dir)
    size = resolved["parameters"]
    if not allow_small_model and not 50_000_000 <= size <= 300_000_000:
        raise ValueError(f"Competition model must have 50M–300M parameters, got {size:,}")
    if size > 300_000_000:
        raise ValueError("Models larger than 300M are never allowed")
    if dry_run:
        print(json.dumps(resolved, indent=2))
        return resolved
    output = Path(output_dir)
    if output.exists() and any(output.iterdir()) and not (output / "last.pt").exists():
        if (output / "status.json").exists() and read_json(output / "status.json").get("status") == "completed":
            raise FileExistsError('Completed model-only run has no optimizer checkpoint; retain it and choose a new output path')
        quarantine_directory(output, "Run has no resumable checkpoint; restarting from initialization")
        resume = False
    if output.exists() and any(output.iterdir()) and not resume:
        raise FileExistsError(f"Run exists: {output}; use --resume or a new output path")
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    settings = resolved["training"]
    seed_everything(settings["seed"])
    dtype_name, dtype = precision(device, settings["precision"])
    train_data, val_data = TokenCorpus(data_dir), TokenCorpus(data_dir, "validation")
    # Verify actual token bytes before committing a long run or resuming one.
    for split in ["train", "validation"]:
        expected = train_data.metadata[f"{split}_sha256"]
        if file_hash(Path(data_dir) / f"{split}.bin") != expected:
            raise ValueError(f"{split} token fingerprint changed")
    from tokenizers import Tokenizer
    tokenizer_path = Path(data_dir) / "tokenizer.json"
    if file_hash(tokenizer_path) != train_data.metadata["tokenizer_sha256"]:
        raise ValueError("Tokenizer fingerprint changed")
    bos_id = Tokenizer.from_file(str(tokenizer_path)).token_to_id("<bos>")
    model = build_model(resolved["model"]).to(device)
    assert sum(p.numel() for p in model.parameters()) == size
    decayed = [p for p in model.parameters() if p.ndim >= 2]
    nondecayed = [p for p in model.parameters() if p.ndim < 2]
    optimizer = torch.optim.AdamW([{"params": decayed, "weight_decay": settings["weight_decay"]},
                                  {"params": nondecayed, "weight_decay": 0.0}],
                                 lr=settings["learning_rate"], betas=(0.9, 0.95), fused=device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=dtype_name == "fp16", init_scale=1024.0)
    rng = np.random.default_rng(settings["seed"])
    first_step, best_loss, cumulative_train_seconds, targets_seen = 0, math.inf, 0.0, 0
    previous_wall_seconds = 0.0
    if resume:
        path = output / "last.pt"
        if not path.exists():
            raise FileNotFoundError(f"Resume checkpoint absent: {path}")
        state = torch.load(path, map_location=device, weights_only=False)
        code_changed = check_resume(state["resolved"], resolved, allow_code_change)
        if code_changed:
            write_json(output / "resume_code_change.json", {"previous": state["resolved"]["training_code_fingerprint"],
                       "current": resolved["training_code_fingerprint"], "step": state["step"],
                       "authorized_by": "--allow-code-change; controlled equivalence is not claimed"})
        if state.get("precision") != dtype_name or state.get("device_type") != device.type:
            raise ValueError("Resume precision/backend changed; resume on the same backend and precision")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        first_step, best_loss = state["step"], state["best_loss"]
        targets_seen, cumulative_train_seconds = state["targets_seen"], state["train_seconds"]
        previous_wall_seconds = state.get("wall_seconds", cumulative_train_seconds)
        rng.bit_generator.state = state["numpy_generator"]
        torch.set_rng_state(state["torch_rng"].cpu())
        if device.type == "cuda":
            torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda_rng"]])
        random.setstate(state["python_rng"])
        if first_step >= resolved["total_steps"]:
            raise ValueError("Run already completed; use a new output path for another experiment")
    write_json(output / "resolved_config.json", resolved)
    if not resume:
        shutil.copyfile(tokenizer_path, output / "tokenizer.json")
        write_json(output / "environment.json", environment())
    write_json(output / "status.json", {"status": "running", "step": first_step,
                                       "started_utc": datetime.now(timezone.utc).isoformat()})
    session_start = time.perf_counter()
    start_train_seconds = cumulative_train_seconds
    model.train()
    step = first_step
    try:
        for step in range(first_step + 1, resolved["total_steps"] + 1):
            warmup = max(1, int(resolved["total_steps"] * settings["warmup_fraction"]))
            if step <= warmup:
                multiplier = step / warmup
            else:
                progress = (step-warmup) / max(1, resolved["total_steps"]-warmup)
                multiplier = settings["min_lr_ratio"] + (1-settings["min_lr_ratio"]) * (1+math.cos(math.pi*progress))/2
            for group in optimizer.param_groups:
                group["lr"] = settings["learning_rate"] * multiplier
            if device.type == "cuda":
                torch.cuda.synchronize()
            tick = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            total_loss, target_count = torch.zeros((), device=device), 0
            sequence_length = settings["sequence_length"]
            if settings["long_sequence_length"] and rng.random() < settings["long_sequence_probability"]:
                sequence_length = settings["long_sequence_length"]
            accumulation = resolved["tokens_per_step"] // (sequence_length * settings["micro_batch_size"])
            length = sequence_length + (model.config.architecture != "encoder_decoder")
            for _ in range(accumulation):
                tokens = torch.from_numpy(train_data.sample(rng, settings["micro_batch_size"], length)).to(device)
                with amp_context(device, dtype):
                    cut = source_cut(settings, length, rng) if model.config.architecture == "encoder_decoder" else None
                    loss, count = loss_for_block(model, tokens, bos_id, settings["source_fraction"], cut)
                total_loss += loss.detach()
                target_count += count
                scaler.scale(loss / accumulation).backward()
            mean_loss = float(total_loss / accumulation)
            if not math.isfinite(mean_loss):
                raise FloatingPointError("Nonfinite training loss")
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"])
            finite_gradients = math.isfinite(float(grad_norm))
            if not finite_gradients and dtype_name != "fp16":
                raise FloatingPointError("Nonfinite gradient norm; retry from checkpoint with bf16/fp32 or lower LR")
            # fp16 overflow is handled by GradScaler's skipped update and scale
            # reduction. Tokens consumed by a skipped update remain logged.
            scaler.step(optimizer)
            scaler.update()
            if device.type == "cuda":
                torch.cuda.synchronize()
            cumulative_train_seconds += time.perf_counter() - tick
            targets_seen += target_count
            stop = ((stop_after_step is not None and step >= stop_after_step) or
                    (max_hours is not None and time.perf_counter()-session_start >= max_hours*3600))
            row = {"step": step, "input_tokens_seen": step * resolved["tokens_per_step"],
                   "target_tokens_seen": targets_seen, "train_loss": mean_loss,
                   "sequence_length": sequence_length, "gradient_accumulation": accumulation,
                   "learning_rate": optimizer.param_groups[0]["lr"],
                   "optimizer_updated": finite_gradients,
                   "train_tokens_per_second": (step-first_step)*resolved["tokens_per_step"] /
                   max(cumulative_train_seconds-start_train_seconds, 1e-9),
                   "train_seconds": cumulative_train_seconds,
                   "peak_memory_bytes": torch.cuda.max_memory_allocated() if device.type == "cuda" else None}
            save = step == first_step+1 or step % settings["eval_every"] == 0 or step == resolved["total_steps"] or stop
            if save:
                row.update(evaluate(model, val_data, settings, bos_id, device, dtype))
                atomic_torch_save({"model": model.state_dict(), "resolved": resolved,
                                   "step": step, "validation_loss": row["validation_loss"]}, output / "latest.pt")
                if row["validation_loss"] < best_loss:
                    best_loss = row["validation_loss"]
                    atomic_torch_save({"model": model.state_dict(), "resolved": resolved,
                                       "step": step, "validation_loss": best_loss}, output / "best.pt")
                atomic_torch_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                   "scaler": scaler.state_dict(), "resolved": resolved, "step": step,
                                   "precision": dtype_name, "device_type": device.type,
                                   "best_loss": best_loss, "targets_seen": targets_seen,
                                   "train_seconds": cumulative_train_seconds,
                                   "wall_seconds": previous_wall_seconds+time.perf_counter()-session_start,
                                   "numpy_generator": rng.bit_generator.state, "python_rng": random.getstate(),
                                   "torch_rng": torch.get_rng_state(),
                                   "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else []}, output / "last.pt")
            with (output / "metrics.jsonl").open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(row, allow_nan=False) + "\n")
            if save or step % 10 == 0:
                print(row, flush=True)
            if stop:
                break
    except Exception as error:
        write_json(output / "status.json", {"status": "failed", "step": step, "error": str(error),
                                           "resume_checkpoint": str(output / "last.pt")})
        raise
    result = {"status": "completed" if step == resolved["total_steps"] else "paused",
              "run_id": resolved["run_id"], "step": step, "best_validation_loss": best_loss,
              "best_checkpoint": str((output / "best.pt").resolve()),
              "input_tokens_seen": step*resolved["tokens_per_step"], "target_tokens_seen": targets_seen,
              "train_seconds": cumulative_train_seconds, "session_seconds": time.perf_counter()-session_start,
              "wall_seconds": previous_wall_seconds+time.perf_counter()-session_start,
              "precision": dtype_name}
    write_json(output / "status.json", result)
    print(result)
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True)
    parser.add_argument("--data", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--allow-code-change", action="store_true", help="Audit opt-in; settings/data/model must still match")
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--stop-after-step", type=int)
    parser.add_argument("--max-hours", type=float)
    parser.add_argument("--allow-small-model", action="store_true", help="Only for non-submittable smoke tests")
    args = vars(parser.parse_args())
    args["config_file"], args["data_dir"], args["output_dir"] = args.pop("config"), args.pop("data"), args.pop("output")
    train(**args)


if __name__ == "__main__":
    main()


### `evaluators/evaluate_phase1.py`


In [ ]:
%%writefile evaluators/evaluate_phase1.py
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)


### `evaluators/provenance.json`


In [ ]:
%%writefile evaluators/provenance.json
{
  "source_url": "https://drive.google.com/file/d/1RgRgm_-u91IBkZVLszaEGRlRzLEwAv2f/view",
  "folder_url": "https://drive.google.com/drive/folders/1cLQw89I1KyBotuY3kgjd1nChs-Msksj4",
  "retrieved_utc": "2026-10-05",
  "sha256": "3f3a9e62be718b478e3f52b378fcc535350ab90b9cb84e3a4c49fec247abf1db",
  "modifications": "none",
  "role": "evaluation only; pretrained scorer is never used to initialize or train Phase 1 LM"
}


### `requirements.txt`


In [ ]:
%%writefile requirements.txt
# Use CUDA-enabled torch in the Jupyter kernel. CPU torch is for offline tests only.
torch>=2.5,<3
numpy>=1.26,<3
tokenizers==0.22.2
huggingface_hub>=0.34,<2
httpx[socks]>=0.28,<1
ijson>=3.3,<4
pytest>=8,<10
# Evaluation only: the supplied official BERTScore scorer downloads its model.
bert-score==0.3.13
transformers>=4.38,<5


### `configs/dense_gpt.json`


In [ ]:
%%writefile configs/dense_gpt.json
{
  "run_id": "dense_gpt_20m_s42_jupyter24gb",
  "model": {
    "architecture": "dense_gpt",
    "dim": 768,
    "layers": 12,
    "heads": 12,
    "kv_heads": 2,
    "ff_dim": 2048,
    "max_context": 2048,
    "encoder_layers": 6,
    "decoder_layers": 6,
    "local_window": 256,
    "query_chunk": 64,
    "global_every": 4,
    "dropout": 0.0,
    "gradient_checkpointing": true
  },
  "training": {
    "seed": 42,
    "sequence_length": 2048,
    "micro_batch_size": 2,
    "gradient_accumulation": 8,
    "token_budget": 20000000,
    "learning_rate": 0.0003,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.03,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_batches": 8,
    "source_fraction": 0.75,
    "precision": "auto"
  }
}


### `configs/encoder_decoder.json`


In [ ]:
%%writefile configs/encoder_decoder.json
{
  "run_id": "encoder_decoder_20m_s42_jupyter24gb",
  "model": {
    "architecture": "encoder_decoder",
    "dim": 768,
    "layers": 12,
    "heads": 12,
    "kv_heads": 2,
    "ff_dim": 2048,
    "max_context": 2048,
    "encoder_layers": 6,
    "decoder_layers": 6,
    "local_window": 256,
    "query_chunk": 64,
    "global_every": 4,
    "dropout": 0.0,
    "gradient_checkpointing": true
  },
  "training": {
    "seed": 42,
    "sequence_length": 2048,
    "micro_batch_size": 2,
    "gradient_accumulation": 8,
    "token_budget": 20000000,
    "learning_rate": 0.0003,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.03,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_batches": 8,
    "source_fraction": 0.75,
    "precision": "auto",
    "source_fraction_range": [
      0.3,
      0.8
    ]
  }
}


### `configs/gqa_local.json`


In [ ]:
%%writefile configs/gqa_local.json
{
  "run_id": "gqa_local_20m_s42_jupyter24gb",
  "model": {
    "architecture": "gqa_local",
    "dim": 768,
    "layers": 14,
    "heads": 12,
    "kv_heads": 2,
    "ff_dim": 2048,
    "max_context": 2048,
    "encoder_layers": 6,
    "decoder_layers": 6,
    "local_window": 256,
    "query_chunk": 64,
    "global_every": 4,
    "dropout": 0.0,
    "gradient_checkpointing": true
  },
  "training": {
    "seed": 42,
    "sequence_length": 2048,
    "micro_batch_size": 2,
    "gradient_accumulation": 8,
    "token_budget": 20000000,
    "learning_rate": 0.0003,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.03,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_batches": 8,
    "source_fraction": 0.75,
    "precision": "auto"
  }
}


### `configs/recurrent_hybrid.json`


In [ ]:
%%writefile configs/recurrent_hybrid.json
{
  "run_id": "recurrent_hybrid_20m_s42_jupyter24gb",
  "model": {
    "architecture": "recurrent_hybrid",
    "dim": 768,
    "layers": 10,
    "heads": 12,
    "kv_heads": 2,
    "ff_dim": 2048,
    "max_context": 2048,
    "encoder_layers": 6,
    "decoder_layers": 6,
    "local_window": 256,
    "query_chunk": 64,
    "global_every": 4,
    "dropout": 0.0,
    "gradient_checkpointing": true
  },
  "training": {
    "seed": 42,
    "sequence_length": 2048,
    "micro_batch_size": 2,
    "gradient_accumulation": 8,
    "token_budget": 20000000,
    "learning_rate": 0.0003,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.03,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_batches": 8,
    "source_fraction": 0.75,
    "precision": "auto"
  }
}


## 2. Use the notebook kernel's CUDA torch and inspect both 24 GB GPUs
PyTorch 2.5+ is sufficient. `auto` precision uses fp16 on T4/P100 and native
bf16 on Ampere or newer. The tiny GQA probe checks the actual installed API.


In [ ]:
import subprocess, torch
from packaging.version import Version
if not Version('2.5') <= Version(torch.__version__.split('+')[0]) < Version('3'):
    raise RuntimeError(f'Require CUDA torch >=2.5,<3; found {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('This Jupyter kernel needs a CUDA-enabled PyTorch installation and NVIDIA driver')
dependencies = [line.strip() for line in Path('requirements.txt').read_text().splitlines()
                if line.strip() and not line.startswith('#') and not line.startswith('torch')]
subprocess.run([sys.executable,'-m','pip','install','--quiet',*dependencies], check=True)
from phase1.train import precision
from phase1.scheduler import visible_gpu_ids, run_jobs
from phase1.runtime import gpu_preflight, accumulation
gpu_records = gpu_preflight(count=2, minimum_vram_gib=22)
print(gpu_records)
for i in range(2):
    device = torch.device(f'cuda:{i}')
    mode, dtype = precision(device)
    q = torch.zeros(1,2,2,8,device=device,dtype=dtype or torch.float32)
    k = torch.zeros(1,1,2,8,device=device,dtype=dtype or torch.float32)
    torch.nn.functional.scaled_dot_product_attention(q,k,k,enable_gqa=True)
    print(i, torch.cuda.get_device_name(i), 'precision:', mode)
del q, k
torch.cuda.empty_cache()


## 3. Scope and deadline
Four architectures are selected by default. Budgets are computed after 50 updates
per architecture. Keep the tag/settings stable when resuming; change the tag for
controlled ablations. Raw data, scoring downloads and models use persistent local folders under `WORK`.
There is no Kaggle-specific disk quota; actual free space is checked. Two models
run concurrently, each on its own GPU. Set `CS781_WORK_DIR` before the first cell
if a larger data drive is available. Both child processes use this kernel's Python.


In [ ]:
MODE = 'full'                          # 'full' or 'profile'; profiling never submits
ARCHITECTURES = ['dense_gpt','gqa_local','encoder_decoder','recurrent_hybrid']
EXPERIMENT_TAG = 'jupyter24gb_v1'
PROFILE_STEPS = 50
HOURS_PER_MODEL = 3.5
BUDGET_SAFETY = 0.85
TRAIN_MICRO_BATCH_SIZE = 2              # Per GPU; reduce to 1 if the measured peak is too large
TOKENS_PER_UPDATE = 32768               # Per model; preserved when changing micro-batch size
GENERATION_BATCH_SIZE = 8               # Per GPU, exact-length buckets; reduce to 4 or 2 for headroom
SCREEN_EXAMPLES = 16                   # Balanced common set; winner confirmed on all held-out examples
RAW_SOURCES = ['train_supreme_court.json','train_high_courts.json']
SOURCE_WEIGHTS = [0.6,0.4]
SEED = 42
ASSIGNMENT1_TOKENIZER = None            # Course-compliant local tokenizer, with four required specials
RUN_SOURCE_ABLATION = False             # Optional extra SC-only dense run, evaluated on the SAME mixed held-out set
USE_HUB = True
LOCAL_TRAIN_PATHS = []
LOCAL_TEST_PATH = None
GPU_IDS = [row['worker_gpu_id'] for row in gpu_records]  # Isolated workers, respects inherited visibility
DATA_ROOT = PROJECT/'data'
RUN_ROOT = PROJECT/'runs'
PROFILE_ROOT = PROJECT/'profiles'
CACHE_ROOT = WORK/'cs781_cache'
RAW_ROOT = CACHE_ROOT/'raw'
os.environ['HF_HOME'] = str(CACHE_ROOT/'huggingface')
SAVED_DISK_LIMIT = None                 # Optional byte quota; filesystem headroom always checked
from datetime import datetime
from zoneinfo import ZoneInfo
DEADLINE = datetime(2026,10,6,23,59,tzinfo=ZoneInfo('Asia/Kolkata'))
if MODE not in {'full','profile'} or PROFILE_STEPS < 50 or HOURS_PER_MODEL <= 0 or TRAIN_MICRO_BATCH_SIZE < 1:
    raise ValueError('Invalid mode/profile/hour settings')
allowed = {'dense_gpt','gqa_local','encoder_decoder','recurrent_hybrid'}
if not ARCHITECTURES or not set(ARCHITECTURES) <= allowed or len(set(ARCHITECTURES)) != len(ARCHITECTURES):
    raise ValueError('Choose distinct supported architectures')
print('GPUs:', GPU_IDS, 'mode:', MODE, 'architectures:', ARCHITECTURES)
print('Hours to deadline:', (DEADLINE-datetime.now(DEADLINE.tzinfo)).total_seconds()/3600)


## 4. Fetch and inspect actual data
The test schema/IDs remain strict. A non-512-word prompt produces a warning and
is retained in full. Empty prompts are errors. Do not invent or truncate prompts.


In [ ]:
from phase1.common import read_json, write_json, file_hash
from phase1.data import inspect, prepare, resolve_official
from phase1.submission import load_test
if USE_HUB:
    from getpass import getpass
    from huggingface_hub import get_token
    hf_token = os.environ.get('HF_TOKEN') or get_token()
    if not hf_token:
        hf_token = getpass('Hugging Face read token (hidden): ').strip()
    if not hf_token:
        raise ValueError('A Hugging Face login/token is needed for the official gated dataset')
    os.environ['HF_TOKEN'] = hf_token
    del hf_token
    source_manifest = resolve_official(DATA_ROOT/'download', RAW_SOURCES, raw_dir=RAW_ROOT)
    train_paths, test_path = source_manifest['train_paths'], source_manifest['test_path']
else:
    if not LOCAL_TRAIN_PATHS or not LOCAL_TEST_PATH:
        raise ValueError('Set attached official paths')
    train_paths = [str(Path(p).resolve()) for p in LOCAL_TRAIN_PATHS]
    test_path = str(Path(LOCAL_TEST_PATH).resolve())
    source_manifest = {'source':'attached official files','train_paths':train_paths,'test_path':test_path}
    write_json(DATA_ROOT/'download'/'source_manifest.json', source_manifest)
inspect(train_paths, limit=3)
official_test = load_test(test_path)
print('Official IDs:', len(official_test))


## 5. Pilot corpus and prompt lengths
Train the tokenizer on training-only documents sampled round-robin from selected
sources, or reuse the permitted Assignment 1 tokenizer. Full preparation reuses
this exact tokenizer. Held-out continuation examples use per-source reservoirs.
If preparation crashes, its files are moved aside and a fresh stage starts.

Use 1024-token training when p99 prompt length plus 250 tokens fits. A 10% mix of
2048-token updates trains the learned position table beyond 1024; effective tokens
per optimizer update stay constant. No prompt is truncated.


In [ ]:
from phase1.profile import prompt_profile
PILOT_DIR = DATA_ROOT/f'pilot_{EXPERIMENT_TAG}'
pilot = prepare(train_paths, PILOT_DIR, train_tokens=5_000_000, val_tokens=2_000_000,
                weights=SOURCE_WEIGHTS, seed=SEED, source_manifest=source_manifest,
                tokenizer_path=ASSIGNMENT1_TOKENIZER, generation_examples=64, reuse=True)
prompt_stats = prompt_profile(test_path, PILOT_DIR/'tokenizer.json')
write_json(PROJECT/'prompt_profile.json', prompt_stats)
print(prompt_stats)
config_root = PROJECT/'run_configs'
config_root.mkdir(exist_ok=True)
def make_config(architecture, run_id, budget, require_corpus=False):
    config = read_json(PROJECT/'configs'/f'{architecture}.json')
    settings = config['training']
    config['run_id'] = run_id
    settings.update(seed=SEED, token_budget=budget, sequence_length=prompt_stats['sequence_length'],
                    micro_batch_size=TRAIN_MICRO_BATCH_SIZE,
                    gradient_accumulation=accumulation(prompt_stats['sequence_length'], TRAIN_MICRO_BATCH_SIZE,
                                                      TOKENS_PER_UPDATE, prompt_stats['long_sequence_length']),
                    long_sequence_length=prompt_stats['long_sequence_length'],
                    long_sequence_probability=prompt_stats['long_sequence_probability'],
                    require_corpus_budget=require_corpus)
    if architecture == 'encoder_decoder':
        # Ensure a realistic encoder prefix and nonempty decoder targets.
        settings['source_length_range'] = [min(v, settings['sequence_length']-128)
                                           for v in prompt_stats['source_length_range']]
    path = config_root/f'{run_id}.json'
    if path.exists() and read_json(path) != config:
        raise ValueError('Config changed under an existing ID; change EXPERIMENT_TAG')
    write_json(path, config)
    return path


## 6. Profile all four architectures on both GPUs
The profile corpus is allowed to be resampled. Profiling checkpoints are isolated
from full runs. Each child sees its assigned GPU as `cuda:0`. Console logs and
scheduler status are saved; an independent successful job survives a peer failure.


In [ ]:
from phase1.train import resolve, check_resume
profile_plans = []
def training_job(config_path, data_dir, run_dir, hours=None):
    current = resolve(config_path, data_dir)
    status = read_json(run_dir/'status.json') if (run_dir/'status.json').exists() else {}
    if (run_dir/'resolved_config.json').exists() and ((run_dir/'last.pt').exists() or status.get('status') == 'completed'):
        check_resume(read_json(run_dir/'resolved_config.json'), current)
    if status.get('status') == 'completed':
        return None
    argv = [sys.executable,'-m','phase1.train','--config',str(config_path),'--data',str(data_dir),'--output',str(run_dir)]
    if (run_dir/'last.pt').exists(): argv.append('--resume')
    if hours is not None: argv += ['--max-hours',str(hours)]
    return {'id':run_dir.name,'argv':argv,'log':str(run_dir.parent/(run_dir.name+'.console.log'))}
for architecture in ARCHITECTURES:
    run_id = f'{architecture}_profile_s{SEED}_{EXPERIMENT_TAG}'
    config_path = make_config(architecture, run_id, PROFILE_STEPS*TOKENS_PER_UPDATE)
    run_dir = PROFILE_ROOT/run_id
    profile_plans.append((architecture, config_path, run_dir))
jobs = [job for _, config, directory in profile_plans
        if (job := training_job(config, PILOT_DIR, directory)) is not None]
run_jobs(jobs, GPU_IDS, PROJECT, PROFILE_ROOT/'training_scheduler.json')


## 7. Benchmark 20 prompts before long training
Undertrained profile models may exhaust all retries. Benchmark mode records short
outputs and timing without packaging a ZIP. Final generation requires every ID
and 128 words. The projection is approximate and includes retries; remeasure final
checkpoints if their behavior changes.


In [ ]:
jobs = []
for architecture, _, directory in profile_plans:
    output = directory/'generation_benchmark'
    if (output/'generation_summary.json').exists(): continue
    argv = [sys.executable,'-m','phase1.generate','--checkpoint',str(directory/'latest.pt'),
            '--test',str(test_path),'--output',str(output),'--benchmark-examples','20',
            '--batch-size',str(GENERATION_BATCH_SIZE)]
    if output.exists(): argv.append('--resume')
    jobs.append({'id':architecture+'_benchmark','argv':argv,'log':str(directory/'generation_benchmark.log')})
run_jobs(jobs, GPU_IDS, PROJECT, PROFILE_ROOT/'generation_scheduler.json')
benchmarks = {a:read_json(d/'generation_benchmark'/'generation_summary.json') for a,_,d in profile_plans}
write_json(PROJECT/'generation_benchmarks.json', benchmarks)
print({a:{'projected_hours':b['projected_full_decode_seconds']/3600,'short_examples':b['short_examples']}
       for a,b in benchmarks.items()})
# Remove redundant PROFILING checkpoint copies only after calibration is complete.
# Keep latest.pt and all metrics/logs; full-training best/last are never touched.
for _, _, directory in profile_plans:
    if (directory/'latest.pt').exists():
        removed = read_json(directory/'released_profile_checkpoints.json') if (directory/'released_profile_checkpoints.json').exists() else []
        for name in ['last.pt','best.pt']:
            path = directory/name
            if path.exists():
                removed.append({'file':name,'sha256':file_hash(path),'reason':'redundant completed profile checkpoint'})
                path.unlink()
        write_json(directory/'released_profile_checkpoints.json', removed)


## 8. Derive budgets and prepare enough unique corpus tokens
The budget uses the smaller measured training/wall throughput and a safety factor.
The full corpus must contain at least the largest planned model budget. Sampling
still uses replacement: processed/stored ratio is recorded and is not an exact
number of epochs. A disk projection includes optimizer states and atomic writes.

`MODE='profile'` skips all remaining training/selection/submission actions.


In [ ]:
from phase1.profile import measured_budget, disk_check
if MODE == 'full':
    budgets = {a:measured_budget(d, HOURS_PER_MODEL, BUDGET_SAFETY) for a,_,d in profile_plans}
    PREPARED_TRAIN_TOKENS = max(b['token_budget'] for b in budgets.values())
    PREPARED_VAL_TOKENS = 5_000_000
    sizes = [read_json(d/'resolved_config.json')['parameters'] for _,_,d in profile_plans]
    disk_projection = disk_check(PROJECT, sizes, PREPARED_TRAIN_TOKENS, PREPARED_VAL_TOKENS,
                                 active_jobs=len(GPU_IDS), limit_bytes=SAVED_DISK_LIMIT,
                                 token_bytes=2 if pilot['dtype']=='uint16' else 4)
    projected_training_hours = HOURS_PER_MODEL*((len(ARCHITECTURES)+len(GPU_IDS)-1)//len(GPU_IDS))
    projected_generation_hours = sum(b['projected_full_decode_seconds'] for b in benchmarks.values())/3600
    hours_left = (DEADLINE-datetime.now(DEADLINE.tzinfo)).total_seconds()/3600
    plan = {'budgets':budgets,'disk':disk_projection,'hours_left':hours_left,
            'projected_training_hours':projected_training_hours,
            'serial_generation_hours':projected_generation_hours,
            'selection_note':'also reserve time for held-out generation, scorer downloads, preprocessing and Codabench queues'}
    write_json(PROJECT/'compute_plan.json', plan)
    print(plan)
    if projected_training_hours+projected_generation_hours+3 > hours_left:
        raise RuntimeError('Plan exceeds deadline reserve; reduce HOURS_PER_MODEL or generation cost and use a new tag')
    TOKEN_DIR = DATA_ROOT/f'mixed_{PREPARED_TRAIN_TOKENS}_s{SEED}_{EXPERIMENT_TAG}'
    data_metadata = prepare(train_paths, TOKEN_DIR, train_tokens=PREPARED_TRAIN_TOKENS,
                            val_tokens=PREPARED_VAL_TOKENS, weights=SOURCE_WEIGHTS, seed=SEED,
                            source_manifest=source_manifest, tokenizer_path=PILOT_DIR/'tokenizer.json',
                            generation_examples=64, reuse=True)
    if data_metadata['counts']['train'] < PREPARED_TRAIN_TOKENS:
        raise ValueError('Sources exhausted below full budget; revise the plan before training')
    heldout = read_json(TOKEN_DIR/'validation_examples.json')
    if not heldout: raise ValueError('No held-out continuation references')
    write_json(PROJECT/'heldout_source_counts.json', {str(i):sum(r['source_id']==i for r in heldout.values())
                                                   for i in range(len(train_paths))})
else:
    print('Profile complete; full-run actions skipped and no submission archives produced')


## 9. Train four full models concurrently, two at a time
Run IDs encode measured budgets. Incomplete directories without a checkpoint are
preserved and restarted. Compatible checkpoints resume their optimizer and RNG.
Training settings/data changes require a new run. Changes to inference-only files
do not block training resume; critical code changes need the explicit CLI audit opt-in.


In [ ]:
plans = []
if MODE == 'full':
    for architecture in ARCHITECTURES:
        budget = budgets[architecture]['token_budget']
        run_id = f'{architecture}_full_{budget}_s{SEED}_{EXPERIMENT_TAG}'
        config = make_config(architecture, run_id, budget, require_corpus=True)
        directory = RUN_ROOT/run_id
        print(run_id, resolve(config,TOKEN_DIR)['parameters'], 'parameters')
        plans.append((architecture,config,directory))
    jobs = [job for _,config,directory in plans
            if (job := training_job(config,TOKEN_DIR,directory,HOURS_PER_MODEL)) is not None]
    run_jobs(jobs, GPU_IDS, PROJECT, RUN_ROOT/'training_scheduler.json')


## 10. Official checkpoint and decoding selection
The unchanged supplied `evaluate_phase1.py` uses `bert-score`, English defaults,
and batch size 8. It downloads a pretrained scoring model solely for evaluation.
The selection stage compares best-loss and latest checkpoints (deduplicated by
step), each with three decoders on a balanced common 16-example screen. The winner
is confirmed on the complete held-out set. All candidate outputs/errors/scores
are retained. Training-objective perplexity is not compared across architectures.


In [ ]:
if MODE == 'full':
    jobs = []
    for architecture,_,directory in plans:
        argv = [sys.executable,'-m','phase1.select','--run',str(directory),
                '--examples',str(TOKEN_DIR/'validation_examples.json'),
                '--batch-size',str(GENERATION_BATCH_SIZE),'--screen-examples',str(SCREEN_EXAMPLES)]
        jobs.append({'id':architecture+'_selection','argv':argv,'log':str(directory/'selection.log')})
    run_jobs(jobs, GPU_IDS, PROJECT, RUN_ROOT/'selection_scheduler.json')
    print({a:read_json(d/'selection.json')['official_bertscore'] for a,_,d in plans})


## 11. Generate and validate the four official archives
Complete prompts are retained. Short/stalled generations retry twice with seeded
sampling; fallback settings are logged. Other errors remain errors. Successful
predictions survive interruption. Every final ZIP contains only root `prediction.json`
with the exact official ID set and 128 words per continuation.


In [ ]:
from IPython.display import display, FileLink
def download_link(path):
    path = Path(path).resolve()
    print('Saved:', path)
    previous_directory = Path.cwd()
    try:
        os.chdir(WORK)
        display(FileLink(str(path.relative_to(WORK))))
    finally:
        os.chdir(previous_directory)
if MODE == 'full':
    jobs, submissions = [], []
    for architecture,_,directory in plans:
        selection = read_json(directory/'selection.json')
        output = directory/'official_selected'
        argv = [sys.executable,'-m','phase1.generate','--checkpoint',selection['checkpoint'],
                '--test',str(test_path),'--output',str(output),'--batch-size',str(GENERATION_BATCH_SIZE)]
        for key,value in selection['decoding'].items():
            if key != 'words': argv += ['--'+key.replace('_','-'),str(value)]
        if output.exists(): argv.append('--resume')
        jobs.append({'id':architecture+'_official','argv':argv,'log':str(directory/'official_generation.log')})
        submissions.append((architecture,output/'submission.zip'))
    run_jobs(jobs, GPU_IDS, PROJECT, RUN_ROOT/'official_scheduler.json')
    hashes = set()
    for architecture,archive in submissions:
        subprocess.run([sys.executable,'-m','phase1.submission','--test',str(test_path),'--zip',str(archive)],check=True)
        hashes.add(file_hash(archive.parent/'prediction.json'))
        print(architecture)
        download_link(archive)
    if len(hashes) != len(submissions):
        raise ValueError('Identical prediction archives; investigate before claiming distinct submissions')
    subprocess.run([sys.executable,'-m','phase1.experiments','ledger','--runs',str(RUN_ROOT),
                    '--output',str(PROJECT/'experiment_ledger.csv')],check=True)
    download_link(PROJECT/'experiment_ledger.csv')


## 12. Optional source-mixing study after securing the four submissions
This extra SC-only dense run shares tokenizer, model, seed, token budget and mixed
held-out evaluation with the main dense run. Only training source composition
changes. Leave disabled if the remaining compute does not fit the deadline.
Record it as planned until it actually runs; it is not part of the four architectures.


In [ ]:
if MODE == 'full' and RUN_SOURCE_ABLATION:
    if len(train_paths) < 2: raise ValueError('Source study requires a multi-source main run')
    budget = budgets['dense_gpt']['token_budget']
    ablation_data = DATA_ROOT/f'sc_only_{budget}_{EXPERIMENT_TAG}'
    prepare(train_paths[:1],ablation_data,train_tokens=budget,val_tokens=5_000_000,
            seed=SEED,tokenizer_path=PILOT_DIR/'tokenizer.json',source_manifest=source_manifest,reuse=True)
    ablation_id = f'dense_gpt_sc_only_{budget}_s{SEED}_{EXPERIMENT_TAG}'
    config = make_config('dense_gpt',ablation_id,budget,require_corpus=True)
    directory = RUN_ROOT/ablation_id
    job = training_job(config,ablation_data,directory,HOURS_PER_MODEL)
    if job: run_jobs([job],GPU_IDS[:1],PROJECT,RUN_ROOT/'source_ablation_scheduler.json')
    run_jobs([{'id':'source_ablation_selection',
               'argv':[sys.executable,'-m','phase1.select','--run',str(directory),
                       '--examples',str(TOKEN_DIR/'validation_examples.json'),
                       '--batch-size',str(GENERATION_BATCH_SIZE),'--screen-examples',str(SCREEN_EXAMPLES)],
               'log':str(directory/'selection.log')}],GPU_IDS[:1],PROJECT)
    write_json(PROJECT/'source_ablation_comparison.json', {
        'changed_factor':'SC-only versus SC+HC training; same tokenizer and held-out prompts',
        'sc_only':read_json(directory/'selection.json'),
        'mixed':read_json(next(d for a,_,d in plans if a=='dense_gpt')/'selection.json')})


## Preserve evidence and submit on Codabench
Save all four full models/tokenizer, configurations, source manifests, input and
supervised target counts, profile measurements, candidate generations, official
scores, fallback diagnostics and actual Codabench submission IDs. Encoder–decoder
supervision differs: report target counts and runtime beside input-token budgets.

Submit early enough for 15–30 minute evaluations and queue delays; these links do
not submit automatically. Infrastructure tests use synthetic CPU fixtures. Actual
24 GB GPU throughput, legal quality and official scores are measured in this notebook.
If a download link is blocked, use Jupyter's file browser to download the printed path.
The code ZIP includes an issue-by-issue patch audit and report outline.
